# Track Data v2: Thu thap Da Chinh sach + Kiem dinh Chat luong Chuan SOTA (Model-Neutral)

Notebook nay chay thuan tren **CPU** (tren may ca nhan hoac Kaggle CPU) de tao bo du lieu v2 giai quyet 5 diem yeu cua Wave 0:
1. **Da hanh vi (Multi-Policy Coverage - arXiv:2408.07986):** `EmptyDayConsistentPolicy` (Anchor) + `Orthogonal Single-Axis Explore` + `PIDWaterResetPolicy`.
2. **Kich thich doc lap (`VIF` check):** Chi nhich 1 truc hanh dong moi block (`30 phut`) thay vi giat cung luc ca 4 truc.
3. **POMDP & Hinh hoc day du:** Bo sung `ep_id`, `step_idx`, `dow_sin/cos`, `zone_ua_norm`, `dT_30m`, `d_oat_30m`, `temp_spread_ahu`.
4. **Chu ky 7 ngay day du (`STEPS_COLLECT = 2016`):** Bao phu ca Thu Tu -> Thu Sau va chuyen giao cuoi tuan (`0-864` vs `864-2016`).
5. **Phan ra BOPTEST KPI:** Luu rieng `comfort_regret`, `energy_cost`, `carbon_emission` va tu xuat bao cao audit (`data_audit_v2_report.png`).


In [ ]:
# Cell 0: Dong bo ma nguon vao /kaggle/working (neu dang chay tren Kaggle)
from pathlib import Path
if Path('/kaggle/working').exists():
    (Path('/kaggle/working') / 'physics_foundation.py').write_text('"""Physics Foundation: TransferPolicy -> EmptyDayConsistentPolicy\nAttribution: Daniel Wu, jazivxt (Facilities V42), penguin069 (Deep Setback)\nSource: https://www.kaggle.com/code/penguin069/smart-buildings-empty-day-consistent-model\nNo neural weights. Only metadata and observations enter decisions.\n"""\nimport math\nimport re\nfrom collections import deque\nimport numpy as np\n\n\nclass TransferPolicy:\n    \'A transferable controller implementing the official two-method API.\'\n    name = \'adaptive_physics\'\n    DEFAULTS = {\n        \'comfort_margin\': 0.12, \'precondition_hours\': 1.0, \'recovery_steps\': 1.5,\n        \'forgetting_factor\': 0.997, \'model_residual_gain\': 0.16, \'comfort_cost\': 65000.0,\n        \'gas_cost_ratio\': 0.30, \'hot_water_target\': 345.0, \'standby_minimum\': 282.0,\n        \'standby_maximum\': 307.0, \'observation_delay\': 1, \'response_scale\': 0.75,\n        \'disturbance_rate\': 0.35,\n    }\n    AIR_CAPACITY = 1.2 * 1006.0\n    WATER_VALVE_CAPACITY = 4180.0 * 1000.0 * 0.108 / 3600.0\n    DESIGN_PRESSURE = 2000.0\n    MIN_PRESSURE = 200.0\n    MIN_DAMPER = 0.10\n    MAX_RECIRCULATION = 0.70\n    MAX_DISCHARGE_K = 308.15\n\n    def __init__(self, building_meta_data, action_spec=None, parameters=None):\n        self.metadata = building_meta_data\n        self.action_spec = action_spec\n        self.parameters = dict(self.DEFAULTS)\n        if parameters:\n            self.parameters.update(parameters)\n        self.n_continuous = int(building_meta_data[\'n_continuous_action\'])\n        self.n_discrete = int(building_meta_data[\'n_discrete_action\'])\n        self.field_names = [str(x) for x in building_meta_data[\'observation_field_names\']]\n        self.field_index = {name: i for i, name in enumerate(self.field_names)}\n        self.obs_norm = building_meta_data.get(\'observation_normalization_metadata\', {})\n        self.action_norm = building_meta_data.get(\'action_normalization_metadata\', {})\n        self.temp_indices = list(building_meta_data[\'temp_indices\'])\n        self.pressure_indices = list(building_meta_data[\'pressure_indices\'])\n        self.recirculation_indices = list(building_meta_data[\'recirculation_indices\'])\n        self.water_indices = list(building_meta_data[\'water_indices\'])\n        self.num_ahus = len(self.temp_indices)\n        zone_actions = building_meta_data[\'zone_action_indices\']\n        self.zone_ids = sorted(zone_actions, key=self._natural_key)\n        self.n_zones = len(self.zone_ids)\n        self.zone_number = {name: i for i, name in enumerate(self.zone_ids)}\n        self.zone_obs = np.array([\n            self.field_index.get(f\'{z}_zone_air_temperature_sensor\', -1) for z in self.zone_ids\n        ], dtype=int)\n        self.zone_low_obs = np.array([\n            self.field_index.get(f\'{z}_zone_air_heating_temperature_setpoint_sensor\', -1) for z in self.zone_ids\n        ], dtype=int)\n        self.zone_high_obs = np.array([\n            self.field_index.get(f\'{z}_zone_air_cooling_temperature_setpoint_sensor\', -1) for z in self.zone_ids\n        ], dtype=int)\n        self.zone_flow_obs = np.array([\n            self.field_index.get(f\'{z}_supply_air_flowrate_setpoint\', -1) for z in self.zone_ids\n        ], dtype=int)\n        self.damper_indices = np.array([\n            zone_actions[z][\'supply_air_damper_percentage_command\'] for z in self.zone_ids\n        ], dtype=int)\n        self.valve_indices = np.array([\n            zone_actions[z][\'reheat_valve_percentage_command\'] for z in self.zone_ids\n        ], dtype=int)\n        assignments = building_meta_data.get(\'ahu_zone_assignments\', building_meta_data.get(\'aha_zone_assignments\', {}))\n        self.ahu_ids = sorted(assignments, key=self._natural_key)\n        self.groups = []\n        self.zone_ahu = np.zeros(self.n_zones, dtype=int)\n        self.outside_obs = []\n        for i, ahu in enumerate(self.ahu_ids):\n            group = np.array([\n                self.zone_number[str(room).replace(\'room_\', \'VAV_\')]\n                for room in assignments[ahu]\n                if str(room).replace(\'room_\', \'VAV_\') in self.zone_number\n            ], dtype=int)\n            self.groups.append(group)\n            self.zone_ahu[group] = i\n            self.outside_obs.append(self._find_field(\n                f\'ahs_{ahu}_outside_air_temperature_sensor\',\n                f\'{ahu}_outside_air_temperature_sensor\',\n                suffix=\'outside_air_temperature_sensor\'))\n        self.water_sensor = self._find_field(suffix=\'supply_water_temperature_sensor\')\n        self.comfort_now_index = self.field_index.get(\'comfort_mode_now\')\n        self.comfort_soon_index = self.field_index.get(\'comfort_mode_soon\')\n        self.occupancy_index = self.field_index.get(\'num_occupants\')\n        self.comfort_band = tuple(map(float, building_meta_data[\'comfort_temp_window\']))\n        self.eco_band = tuple(map(float, building_meta_data[\'eco_temp_window\']))\n        self.start_hour = float(building_meta_data.get(\'morning_start_hour\', 6))\n        self.end_hour = float(building_meta_data.get(\'evening_start_hour\', 19))\n        self.zone_area, self.zone_ua = self._geometry_priors(building_meta_data.get(\'zone_map\'))\n        effective_capacity = 3800.0 * self.zone_area + 21000.0 * np.sqrt(self.zone_area)\n        self.initial_gain = np.clip(300000.0 / effective_capacity, 0.08, 2.5)\n        self.reset()\n\n    @staticmethod\n    def _natural_key(value):\n        return tuple(int(x) if x.isdigit() else x for x in re.split(r\'(\\d+)\', str(value)))\n\n    def _find_field(self, *exact, suffix=\'\'):\n        for name in exact:\n            if name in self.field_index:\n                return self.field_index[name]\n        if suffix:\n            for name, index in self.field_index.items():\n                if name.endswith(suffix):\n                    return index\n        return None\n\n    def _geometry_priors(self, zone_map):\n        areas = np.full(self.n_zones, 55.0)\n        ua = np.full(self.n_zones, 18.0)\n        if zone_map is None:\n            return areas, ua\n        grid = np.asarray(zone_map)\n        if grid.ndim != 2 or grid.size > 5000000:\n            return areas, ua\n        height, width = grid.shape\n        inside = (grid == 0).ravel()\n        visited = np.zeros(grid.size, dtype=bool)\n        flat = grid.ravel()\n        counts, external_edges = [], []\n        for start in np.flatnonzero(inside):\n            if visited[start]:\n                continue\n            stack = [int(start)]\n            visited[start] = True\n            count, edge = 0, 0\n            while stack:\n                p = stack.pop()\n                count += 1\n                row, col = divmod(p, width)\n                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                    nr, nc = row + dr, col + dc\n                    if not (0 <= nr < height and 0 <= nc < width):\n                        continue\n                    q = nr * width + nc\n                    if inside[q] and not visited[q]:\n                        visited[q] = True\n                        stack.append(q)\n                    elif flat[q] == 2:\n                        edge += 1\n                    elif flat[q] == 1:\n                        rr, cc = row + 2 * dr, col + 2 * dc\n                        if 0 <= rr < height and 0 <= cc < width and grid[rr, cc] == 2:\n                            edge += 1\n            counts.append(count)\n            external_edges.append(edge)\n        if len(counts) != self.n_zones:\n            return areas, ua\n        for j, name in enumerate(self.zone_ids):\n            match = re.search(r\'(\\d+)$\', name)\n            label = int(match.group(1)) - 1 if match else j\n            if 0 <= label < len(counts):\n                areas[j] = max(1.0, 0.04 * counts[label])\n                ua[j] = max(0.5, external_edges[label] * 0.2 * 3.0 * 1.3)\n        return areas, ua\n\n    def reset(self):\n        self.step_count = 0\n        self.history = deque(maxlen=3)\n        self.previous_observed = None\n        self.slow_temperature = None\n        self.residual = np.zeros(self.n_zones)\n        self.model_error = np.full(self.n_zones, 0.08)\n        self.theta = np.zeros((self.n_zones, 6))\n        self.theta[:, 0] = self.initial_gain * self.zone_area * 2.0 / 1000.0\n        self.theta[:, 1] = self.initial_gain * self.zone_ua * 20.0 / 1000.0\n        self.theta[:, 2] = 0.18\n        self.theta[:, 3] = 0.08\n        self.theta[:, 4] = self.initial_gain * self.zone_area * 8.0 / 1000.0\n        self.theta[:, 5] = self.initial_gain * self.parameters[\'response_scale\']\n        self.theta[:, :2] *= self.parameters[\'response_scale\']\n        self.theta[:, 4] *= self.parameters[\'response_scale\']\n        prior = np.array([0.12, 0.20, 0.15, 0.15, 0.16, 0.8])\n        self.covariance = np.repeat(np.diag(prior)[None, :, :], self.n_zones, axis=0)\n        self.last_heating_step = -1000\n        self.last_action = None\n\n    def _temperature(self, obs, index, kind, fallback):\n        if index is None:\n            return fallback\n        value = float(obs[index])\n        if 150.0 < value < 450.0:\n            return value\n        norm = self.obs_norm.get(kind, {})\n        mean = float(norm.get(\'sample_mean\', fallback))\n        std = float(norm.get(\'sample_std\', math.sqrt(norm.get(\'sample_variance\', 1.0))))\n        value = value * std + mean\n        return value if np.isfinite(value) and 150.0 < value < 450.0 else fallback\n\n    def _zone_temperatures(self, obs):\n        norm = self.obs_norm.get(\'zone_air_temperature_sensor\', {})\n        mean = float(norm.get(\'sample_mean\', 295.0))\n        std = float(norm.get(\'sample_std\', math.sqrt(norm.get(\'sample_variance\', 25.0))))\n        raw = obs[self.zone_obs].astype(float)\n        value = np.where((raw > 150.0) & (raw < 450.0), raw, raw * std + mean)\n        fallback = self.previous_observed if self.previous_observed is not None else mean\n        return np.where(np.isfinite(value) & (value > 150.0) & (value < 450.0), value, fallback)\n\n    def _hour(self, obs):\n        sine = self.field_index.get(\'hod_sin_000\')\n        cosine = self.field_index.get(\'hod_cos_000\')\n        if sine is None or cosine is None:\n            return (self.start_hour + self.step_count / 12.0) % 24.0\n        return (math.atan2(float(obs[sine]), float(obs[cosine])) % (2.0 * math.pi)) * 12.0 / math.pi\n\n    def _band(self, obs, temperature):\n        now = bool(obs[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else (\n            self.start_hour <= self._hour(obs) < self.end_hour)\n        soon = bool(obs[self.comfort_soon_index] > 0.5) if self.comfort_soon_index is not None else now\n        default = self.comfort_band if now else self.eco_band\n        low = np.full(self.n_zones, default[0])\n        high = np.full(self.n_zones, default[1])\n        valid = (self.zone_low_obs >= 0) & (self.zone_high_obs >= 0)\n        if np.any(valid):\n            sensor_low = obs[self.zone_low_obs[valid]].astype(float)\n            sensor_high = obs[self.zone_high_obs[valid]].astype(float)\n            good = (sensor_low > 150.0) & (sensor_high < 450.0) & (sensor_high > sensor_low)\n            indices = np.flatnonzero(valid)[good]\n            low[indices] = sensor_low[good]\n            high[indices] = sensor_high[good]\n        if now:\n            low = np.maximum(low, self.comfort_band[0])\n            high = np.minimum(high, self.comfort_band[1])\n        elif soon:\n            hour = self._hour(obs)\n            remaining = max(0.0, (self.start_hour - hour) % 24.0)\n            progress = np.clip(1.0 - remaining / max(0.1, self.parameters[\'precondition_hours\']), 0.0, 1.0)\n            low = self.eco_band[0] + progress * (self.comfort_band[0] - self.eco_band[0]) + np.zeros(self.n_zones)\n            high = self.eco_band[1] + progress * (self.comfort_band[1] - self.eco_band[1]) + np.zeros(self.n_zones)\n        else:\n            low[:] = self.parameters[\'standby_minimum\']\n            high[:] = self.parameters[\'standby_maximum\']\n        margin = self.parameters[\'comfort_margin\'] if (now or soon) else 0.0\n        low += margin\n        high -= margin\n        recovery = max(1.0, self.parameters[\'recovery_steps\'])\n        low = np.minimum(low, temperature + np.maximum((low - temperature) / recovery, 0.0))\n        high = np.maximum(high, temperature - np.maximum((temperature - high) / recovery, 0.0))\n        return low, high, now\n\n    def _group_temperature(self, temperature):\n        means = np.zeros(self.n_zones)\n        for group in self.groups:\n            if group.size:\n                means[group] = np.average(temperature[group], weights=self.zone_area[group])\n        return means\n\n    def _features(self, temperature, outside, mode, slow):\n        return np.column_stack((\n            np.ones(self.n_zones),\n            np.clip((outside - temperature) / 20.0, -5.0, 5.0),\n            np.clip((slow - temperature) / 5.0, -5.0, 5.0),\n            np.clip((self._group_temperature(temperature) - temperature) / 5.0, -5.0, 5.0),\n            np.full(self.n_zones, float(mode)),\n        ))\n\n    def _learn(self, observed):\n        delay = int(self.parameters[\'observation_delay\'])\n        if self.previous_observed is None or len(self.history) <= delay:\n            return\n        applied = list(self.history)[-(delay + 1)]\n        old = self.previous_observed\n        power = (applied[\'conductance\'] * (applied[\'supply\'] - observed) + applied[\'reheat\']) / 1000.0\n        features = self._features(old, applied[\'outside\'], applied[\'mode\'], applied[\'slow\'])\n        x = np.column_stack((features, power))\n        delta = np.clip(observed - old, -15.0, 15.0)\n        prediction = np.einsum(\'ij,ij->i\', self.theta, x)\n        remaining = delta - prediction\n        rate = float(self.parameters[\'disturbance_rate\'])\n        self.model_error = 0.94 * self.model_error + 0.06 * np.minimum(np.abs(remaining - self.residual), 2.0)\n        self.residual = np.clip((1.0 - rate) * self.residual + rate * remaining, -5.0, 5.0)\n\n    def _free_drift(self, temperature, outside, mode):\n        features = self._features(temperature, outside, mode, self.slow_temperature)\n        return np.clip(np.einsum(\'ij,ij->i\', self.theta[:, :5], features) + self.residual, -5.0, 5.0)\n\n    def _native_action(self, kind, value, low, high):\n        spec = self.action_norm.get(kind)\n        if spec is None:\n            spec = next((v for k, v in self.action_norm.items() if str(k).endswith(kind)), None)\n        if spec:\n            low = float(spec.get(\'min_native_value\', low))\n            high = float(spec.get(\'max_native_value\', high))\n            out_low = float(spec.get(\'min_normalized_value\', -1.0))\n            out_high = float(spec.get(\'max_normalized_value\', 1.0))\n        else:\n            out_low, out_high = -1.0, 1.0\n        return np.clip(out_low + (np.asarray(value) - low) * (out_high - out_low) / max(high - low, 1e-12), out_low, out_high)\n\n    def _choose_ahu(self, group, temperature, free, low, high, max_flow, outside, water):\n        t, d, lo, hi = temperature[group], free[group], low[group], high[group]\n        response = self.theta[group, 5]\n        capacity = max_flow[group]\n        inside = float(np.average(t, weights=self.zone_area[group]))\n        targets = np.array([285.0, 287.0, 289.0, 291.0, 293.0, 295.0, 297.0, 299.0, 302.0, 305.0])\n        targets = np.concatenate((targets, [float(np.clip(np.min(hi) - 1.5, 285.0, 305.0))]))\n        setpoint = np.repeat(targets, 2)\n        recirc = np.full_like(setpoint, self.MAX_RECIRCULATION)\n        if outside < inside - 0.5:\n            recirc[1::2] = np.clip((targets - outside) / max(inside - outside, 1e-6), 0.0, self.MAX_RECIRCULATION)\n        mixed = recirc * inside + (1.0 - recirc) * outside\n        supply = np.minimum(mixed, setpoint)\n        sa = supply[:, None]\n        cool_need = np.maximum(t + d - hi, 0.0) * 1000.0 / response\n        heat_need = np.maximum(lo - t - d, 0.0) * 1000.0 / response\n        cooling_flow = cool_need[None, :] / (self.AIR_CAPACITY * np.maximum(hi[None, :] - sa, 0.05))\n        heating_flow = heat_need[None, :] / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - lo[None, :], 0.5))\n        maximum_water_heat = self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * np.maximum(water - sa, 0.0)\n        maximum_heating_flow = maximum_water_heat / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - sa, 0.1))\n        heating_flow = np.minimum(heating_flow, maximum_heating_flow)\n        needed_flow = np.maximum(cooling_flow, heating_flow)\n        fraction = np.max(needed_flow / capacity[None, :], axis=1)\n        pressure = np.clip(self.DESIGN_PRESSURE * fraction ** 2, self.MIN_PRESSURE, self.DESIGN_PRESSURE)\n        pressure_ratio = np.sqrt(pressure / self.DESIGN_PRESSURE)\n        dampers = np.clip(needed_flow / (capacity[None, :] * pressure_ratio[:, None]), self.MIN_DAMPER, 1.0)\n        flow = dampers * capacity[None, :] * pressure_ratio[:, None]\n        conductance = flow * self.AIR_CAPACITY\n        water_gap = np.maximum(water - sa, 0.0)\n        wanted_reheat = np.maximum(\n            (lo[None, :] - t[None, :] - d[None, :]) * 1000.0 / response[None, :] + conductance * (lo[None, :] - sa), 0.0)\n        reheat_limit = np.minimum(\n            conductance * np.maximum(self.MAX_DISCHARGE_K - sa, 0.0),\n            self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * water_gap)\n        reheat = np.minimum(wanted_reheat, reheat_limit)\n        predicted = (t[None, :] + d[None, :] + response[None, :] * (conductance * sa + reheat) / 1000.0) / (\n            1.0 + response[None, :] * conductance / 1000.0)\n        violation = np.maximum(lo[None, :] - predicted, 0.0) + np.maximum(predicted - hi[None, :], 0.0)\n        thermal_penalty = np.sum(violation + 2.0 * violation ** 2, axis=1)\n        total_flow = flow.sum(axis=1)\n        fan = total_flow * pressure * (2.0 - recirc) / 0.9\n        chiller = total_flow * self.AIR_CAPACITY * np.maximum(mixed - supply, 0.0) / 3.5\n        gas = reheat.sum(axis=1)\n        objective = fan + chiller + self.parameters[\'gas_cost_ratio\'] * gas\n        objective += self.parameters[\'comfort_cost\'] * thermal_penalty\n        off_prediction = t + d\n        off_violation = np.maximum(lo - off_prediction, 0.0) + np.maximum(off_prediction - hi, 0.0)\n        off_objective = self.parameters[\'comfort_cost\'] * np.sum(off_violation + 2.0 * off_violation ** 2)\n        best = int(np.argmin(objective))\n        if off_objective <= objective[best]:\n            return {\'run\': 0, \'pressure\': 0.0, \'setpoint\': 305.0, \'recirc\': 0.7,\n                    \'damper\': np.full(group.size, self.MIN_DAMPER), \'conductance\': np.zeros(group.size),\n                    \'supply\': np.full(group.size, mixed[best]), \'reheat\': np.zeros(group.size)}\n        return {\'run\': 1, \'pressure\': float(pressure[best]), \'setpoint\': float(setpoint[best]),\n                \'recirc\': float(recirc[best]), \'damper\': dampers[best],\n                \'conductance\': conductance[best], \'supply\': np.full(group.size, supply[best]),\n                \'reheat\': reheat[best]}\n\n    def get_action(self, obs):\n        obs = np.asarray(obs, dtype=float).reshape(-1)\n        observed = self._zone_temperatures(obs)\n        if self.slow_temperature is None:\n            self.slow_temperature = observed.copy()\n        self._learn(observed)\n        outside_ahu = np.array([\n            self._temperature(obs, index, \'outside_air_temperature_sensor\', 290.0)\n            for index in self.outside_obs\n        ])\n        outside = outside_ahu[self.zone_ahu]\n        mode = bool(obs[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else True\n        temperature = observed.copy()\n        if self.history and int(self.parameters[\'observation_delay\']) == 1:\n            last = self.history[-1]\n            drift = self._free_drift(observed, last[\'outside\'], last[\'mode\'])\n            gain = self.theta[:, 5] / 1000.0\n            temperature = (observed + drift + gain * (last[\'conductance\'] * last[\'supply\'] + last[\'reheat\'])) / (\n                1.0 + gain * last[\'conductance\'])\n            temperature = np.clip(temperature, observed - 8.0, observed + 8.0)\n        low, high, mode = self._band(obs, temperature)\n        free = self._free_drift(temperature, outside, mode)\n        max_flow = np.full(self.n_zones, 2.0)\n        valid = self.zone_flow_obs >= 0\n        if np.any(valid):\n            flow_reading = obs[self.zone_flow_obs[valid]]\n            max_flow[valid] = np.where(np.isfinite(flow_reading) & (flow_reading > 0.0), flow_reading, 2.0)\n        max_flow = np.clip(max_flow, 0.01, 100.0)\n        water_observed = self._temperature(obs, self.water_sensor, \'supply_water_temperature_sensor\', 340.0)\n        heat_demand = np.any(temperature + free < low)\n        if heat_demand:\n            self.last_heating_step = self.step_count\n        recent_heat = self.step_count - self.last_heating_step < 24\n        water_target = self.parameters[\'hot_water_target\'] if recent_heat else 310.0\n        water_available = water_observed\n        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)\n        discrete = np.zeros(self.n_discrete, dtype=np.int32)\n        conductance = np.zeros(self.n_zones)\n        supply = outside.copy()\n        reheat = np.zeros(self.n_zones)\n        for i, group in enumerate(self.groups):\n            if not group.size:\n                continue\n            choice = self._choose_ahu(group, temperature, free, low, high, max_flow,\n                                      float(outside_ahu[i]), float(water_available))\n            discrete[i] = choice[\'run\']\n            continuous[self.temp_indices[i]] = self._native_action(\n                f\'{self.ahu_ids[i]}_supply_air_temperature_setpoint\', choice[\'setpoint\'], 285.0, 305.0)\n            continuous[self.pressure_indices[i]] = self._native_action(\n                f\'{self.ahu_ids[i]}_supply_air_static_pressure_setpoint\', choice[\'pressure\'], 0.0, 2000.0)\n            continuous[self.recirculation_indices[i]] = self._native_action(\n                f\'{self.ahu_ids[i]}_recirculation_percentage_command\', choice[\'recirc\'], 0.0, 1.0)\n            continuous[self.damper_indices[group]] = self._native_action(\n                \'supply_air_damper_percentage_command\', choice[\'damper\'], 0.0, 1.0)\n            conductance[group] = choice[\'conductance\']\n            supply[group] = choice[\'supply\']\n            reheat[group] = choice[\'reheat\']\n        if np.any(reheat > 10.0):\n            self.last_heating_step = self.step_count\n            water_target = self.parameters[\'hot_water_target\']\n        valve_need = reheat / (self.WATER_VALVE_CAPACITY * np.maximum(water_available - supply, 1.0))\n        pressure_water = float(np.clip(np.max(valve_need) ** 2 * 1.05, 0.0, 20.0))\n        if pressure_water > 1e-8:\n            valves = np.clip(valve_need / math.sqrt(pressure_water), 0.0, 1.0)\n            discrete[-1] = 1\n        else:\n            valves = np.zeros(self.n_zones)\n        continuous[self.valve_indices] = self._native_action(\'reheat_valve_percentage_command\', valves, 0.0, 1.0)\n        if self.water_indices:\n            continuous[self.water_indices[0]] = self._native_action(\n                \'supply_water_temperature_setpoint\', water_target, 310.0, 350.0)\n        if len(self.pressure_indices) > self.num_ahus:\n            continuous[self.pressure_indices[-1]] = self._native_action(\n                \'differential_pressure_setpoint\', pressure_water, 0.0, 20.0)\n        actual_reheat = np.minimum(\n            self.WATER_VALVE_CAPACITY * math.sqrt(pressure_water) * valves * np.maximum(water_available - supply, 0.0),\n            conductance * np.maximum(self.MAX_DISCHARGE_K - supply, 0.0))\n        self.history.append({\'conductance\': conductance, \'supply\': supply,\n                             \'reheat\': actual_reheat, \'outside\': outside.copy(),\n                             \'mode\': mode, \'slow\': self.slow_temperature.copy()})\n        self.slow_temperature += (temperature - self.slow_temperature) / 48.0\n        self.previous_observed = observed.copy()\n        self.step_count += 1\n        action = {\'continuous_action\': np.clip(np.nan_to_num(continuous, nan=-1.0), -1.0, 1.0).astype(np.float32),\n                  \'discrete_action\': discrete.astype(np.int32)}\n        self.last_action = action\n        return action\n\n\nclass GuardedJointThermalPolicy(TransferPolicy):\n    name = \'facilities_v40_joint_thermal\'\n    def _zone_temperatures(self, obs):\n        value = super()._zone_temperatures(obs)\n        previous = self.previous_observed if self.previous_observed is not None else np.mean(self.comfort_band)\n        return np.where(np.isfinite(value) & (value >= 150.) & (value <= 400.), value, previous)\n\n\ndef bounded_rls(theta, covariance, x, target, lower, upper, forgetting=.997):\n    px = np.einsum(\'nij,nj->ni\', covariance, x)\n    denominator = forgetting + np.einsum(\'ni,ni->n\', x, px)\n    gain = px / np.maximum(denominator[:, None], 1e-9)\n    innovation = np.clip(target - np.einsum(\'ni,ni->n\', theta, x), -2., 2.)\n    fitted = np.clip(theta + gain * innovation[:, None], lower, upper)\n    updated = (covariance - np.einsum(\'ni,nj->nij\', gain, px)) / forgetting\n    updated = .5 * (updated + updated.transpose(0, 2, 1))\n    return fitted, updated\n\n\nclass IdentifiedJointThermalPolicy(GuardedJointThermalPolicy):\n    name = \'facilities_v41_identified_thermal\'\n    def reset(self):\n        super().reset()\n        self._thermal_prior = self.theta[:, 5].copy()\n    def _learn(self, observed):\n        delay = int(self.parameters[\'observation_delay\'])\n        if self.previous_observed is None or len(self.history) <= delay:\n            return\n        applied = list(self.history)[-(delay + 1)]\n        power = (applied[\'conductance\'] * (applied[\'supply\'] - observed) + applied[\'reheat\']) / 1000.\n        features = self._features(self.previous_observed, applied[\'outside\'], applied[\'mode\'], applied[\'slow\'])\n        x = np.column_stack((features, power))\n        delta = observed - self.previous_observed\n        valid = np.isfinite(x).all(axis=1) & np.isfinite(delta) & (np.abs(delta) <= 3.)\n        lower = np.zeros_like(self.theta)\n        upper = np.ones_like(self.theta) * np.array([2., 5., 2., 2., 2., 1.])\n        lower[:, 0] = -2.\n        lower[:, 5] = .5 * self._thermal_prior\n        upper[:, 5] = 1.5 * self._thermal_prior\n        if np.any(valid):\n            fit, cov = bounded_rls(self.theta[valid], self.covariance[valid], x[valid],\n                delta[valid] - self.residual[valid], lower[valid], upper[valid],\n                float(self.parameters[\'forgetting_factor\']))\n            self.theta[valid], self.covariance[valid] = fit, cov\n        remaining = np.clip(delta - np.einsum(\'ni,ni->n\', self.theta, x), -5., 5.)\n        self.model_error = .94 * self.model_error + .06 * np.minimum(np.abs(remaining - self.residual), 2.)\n        rate = float(self.parameters[\'disturbance_rate\'])\n        self.residual = np.clip((1 - rate) * self.residual + rate * remaining, -5., 5.)\n\n\nclass ErrorReserveJointThermalPolicy(GuardedJointThermalPolicy):\n    name = \'facilities_v42_error_reserve_thermal\'\n    def _band(self, obs, temperature):\n        low, high, mode = super()._band(obs, temperature)\n        if mode:\n            extra = np.minimum(np.clip(.5 * (self.model_error - .08), 0., .25), .25 * np.maximum(high - low, 0.))\n            low, high = low + extra, high - extra\n        return low, high, mode\n\n\nclass IdentifiedReservePolicy(IdentifiedJointThermalPolicy, ErrorReserveJointThermalPolicy):\n    name = \'identified_reserve\'\n\n\nclass CausalIdentifiedReservePolicy(IdentifiedReservePolicy):\n    name = \'causal_identified\'\n    def _learn(self, observed):\n        delay = int(self.parameters[\'observation_delay\'])\n        if self.previous_observed is None or len(self.history) <= delay:\n            return\n        applied = list(self.history)[-(delay + 1)]\n        old = self.previous_observed\n        features = self._features(old, applied[\'outside\'], applied[\'mode\'], applied[\'slow\'])\n        drift = np.einsum(\'ij,ij->i\', self.theta[:, :5], features) + self.residual\n        gain = self.theta[:, 5] / 1000.0\n        prediction = (old + drift + gain * (applied[\'conductance\'] * applied[\'supply\'] + applied[\'reheat\'])) / (\n            1.0 + gain * applied[\'conductance\'])\n        error = np.where(np.isfinite(observed - prediction), np.minimum(np.abs(observed - prediction), 2.0), 2.0)\n        previous_error = self.model_error.copy()\n        super()._learn(observed)\n        self.model_error = 0.94 * previous_error + 0.06 * error\n\n\nclass WarmStandbyPolicy(CausalIdentifiedReservePolicy):\n    name = \'causal_warm_standby\'\n    HOLD_STEPS = 36 * 12\n    def _native_action(self, kind, value, low, high):\n        if (kind == \'supply_water_temperature_setpoint\'\n                and self.step_count - self.last_heating_step < self.HOLD_STEPS):\n            value = self.parameters[\'hot_water_target\']\n        return super()._native_action(kind, value, low, high)\n\n\nclass WeekendWaterHoldPolicy(WarmStandbyPolicy):\n    name = \'warm_hold72\'\n    HOLD_STEPS = 72 * 12\n\n\nclass DeepFastRecoveryPolicy(WeekendWaterHoldPolicy):\n    name = \'hold72_deep_fast_recovery\'\n    DEFAULTS = dict(WeekendWaterHoldPolicy.DEFAULTS, standby_minimum=274.0, recovery_steps=1.0)\n\n\nclass EmptyDayConsistentPolicy(DeepFastRecoveryPolicy):\n    name = \'deep_empty_day_consistent_model\'\n    EMPTY_CONFIRMATION_HOURS = 2.0\n    def reset(self):\n        super().reset()\n        self.empty_day_relaxed_steps = 0\n    def _band(self, obs, temperature):\n        low, high, schedule_mode = super()._band(obs, temperature)\n        if not schedule_mode or self.occupancy_index is None:\n            return low, high, schedule_mode\n        occupants = float(obs[self.occupancy_index])\n        elapsed = (self._hour(obs) - self.start_hour) % 24.0\n        scheduled_duration = (self.end_hour - self.start_hour) % 24.0\n        if (not np.isfinite(occupants) or occupants != 0.0\n                or not self.EMPTY_CONFIRMATION_HOURS <= elapsed < scheduled_duration):\n            return low, high, schedule_mode\n        self.empty_day_relaxed_steps += 1\n        return (np.full(self.n_zones, self.parameters[\'standby_minimum\']),\n                np.full(self.n_zones, self.parameters[\'standby_maximum\']), schedule_mode)\n')
    (Path('/kaggle/working') / 'warp_resip.py').write_text('"""Stochastic Warp RL (IQL-Warp) + Physics-Uncertainty ResiP Gate Module\nUpgrades over v1:\n1. RationalQuadraticWarp1D: Added hard upper slope bound (max_derivative=2.2)\n   and exact closed-form analytical inverse() for Change-of-Variables log_prob.\n2. DeepSetsWarpResiPActor: Stochastic Pre-Warp Gaussian N(x_base, sigma(s)^2)\n   passed through invertible spline + Physics-Uncertainty-Gated ResiP (modulated\n   by causal RLS model_error) + Antithetic MC inference (exact Identity at init).\n3. DeepSetsValueNet & DeepSetsAHUTwinCritic: Enables Implicit Q-Learning (IQL)\n   so Value/Critic NEVER query out-of-distribution actions during Bellman backups.\n4. WarpResiPMixin & make_warp_policy(base_cls): Physics re-projection of VAV\n   dampers and reheat valves preserving energy-mass conservation.\n"""\nimport math\nimport sys\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n_HERE = str(Path(__file__).resolve().parent)\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\n\nfrom physics_foundation import (\n    CausalIdentifiedReservePolicy,\n    EmptyDayConsistentPolicy,\n    TransferPolicy,\n)\n\nZONE_FEAT_DIM = 9\nGLOBAL_FEAT_DIM = 10\nAHU_ACT_DIM = 4  # [sat_norm, press_norm, recirc_norm, water_norm]\n\n\nclass RationalQuadraticWarp1D(nn.Module):\n    """Bounded-slope Monotonic Rational-Quadratic Spline on [-bound, bound].\n\n    Guarantees exact Identity at zero logits (W(x) == x) and Lipschitz slope bounds\n    d in [min_derivative, max_derivative], with exact analytical forward & inverse.\n    """\n    def __init__(self, num_bins=8, bound=1.05, min_bin_width=0.05,\n                 min_bin_height=0.05, min_derivative=0.40, max_derivative=2.20):\n        super().__init__()\n        self.K = num_bins\n        self.B = float(bound)\n        self.min_w = float(min_bin_width)\n        self.min_h = float(min_bin_height)\n        self.min_d = float(min_derivative)\n        self.max_d = float(max_derivative)\n        assert self.K * self.min_w < 2.0 * self.B\n        assert self.K * self.min_h < 2.0 * self.B\n        assert self.min_d < 1.0 < self.max_d\n        t0 = (1.0 - self.min_d) / (self.max_d - self.min_d)\n        self.c0 = math.log(t0 / (1.0 - t0))\n\n    @property\n    def num_params_per_dim(self):\n        return 3 * self.K - 1\n\n    def _compute_knots(self, raw_params, gate=None):\n        K, B = self.K, self.B\n        raw_w, raw_h, raw_d = torch.split(raw_params, [K, K, K - 1], dim=-1)\n        if gate is not None:\n            raw_w = gate * raw_w\n            raw_h = gate * raw_h\n            raw_d = gate * raw_d\n\n        widths = self.min_w + (2.0 * B - K * self.min_w) * F.softmax(raw_w, dim=-1)\n        heights = self.min_h + (2.0 * B - K * self.min_h) * F.softmax(raw_h, dim=-1)\n\n        pad_lo = torch.full_like(widths[..., :1], -B)\n        pad_hi = torch.full_like(widths[..., :1], B)\n        cumwidths = torch.cat([pad_lo, torch.cumsum(widths, dim=-1)[..., :-1] - B, pad_hi], dim=-1)\n        cumheights = torch.cat([pad_lo, torch.cumsum(heights, dim=-1)[..., :-1] - B, pad_hi], dim=-1)\n\n        interior_d = self.min_d + (self.max_d - self.min_d) * torch.sigmoid(raw_d + self.c0)\n        derivatives = F.pad(interior_d, (1, 1), value=1.0)\n        return widths, heights, cumwidths, cumheights, derivatives\n\n    def forward(self, x, raw_params, gate=None):\n        K, B = self.K, self.B\n        widths, heights, cumwidths, cumheights, derivatives = self._compute_knots(raw_params, gate)\n\n        x_c = x.clamp(-B + 1e-6, B - 1e-6)\n        bin_idx = ((x_c.unsqueeze(-1) >= cumwidths[..., :-1]).sum(dim=-1, keepdim=True) - 1).clamp(0, K - 1)\n\n        xk = cumwidths.gather(-1, bin_idx).squeeze(-1)\n        wk = widths.gather(-1, bin_idx).squeeze(-1)\n        yk = cumheights.gather(-1, bin_idx).squeeze(-1)\n        hk = heights.gather(-1, bin_idx).squeeze(-1)\n        dk = derivatives.gather(-1, bin_idx).squeeze(-1)\n        dk1 = derivatives.gather(-1, bin_idx + 1).squeeze(-1)\n\n        sk = hk / wk.clamp(min=1e-8)\n        theta = ((x_c - xk) / wk.clamp(min=1e-8)).clamp(0.0, 1.0)\n        theta_1m = theta * (1.0 - theta)\n        delta_d = dk1 + dk - 2.0 * sk\n\n        num = hk * (sk * theta.pow(2) + dk * theta_1m)\n        den = (sk + delta_d * theta_1m).clamp(min=1e-8)\n        y_raw = yk + num / den\n\n        grad_num = sk.pow(2) * (dk1 * theta.pow(2) + 2.0 * sk * theta_1m + dk * (1.0 - theta).pow(2))\n        dydx = (grad_num / den.pow(2).clamp(min=1e-12)).clamp(min=1e-6)\n        log_det_j = torch.log(dydx).sum(dim=-1)\n\n        return y_raw.clamp(-1.0, 1.0), y_raw, log_det_j\n\n    def inverse(self, y, raw_params, gate=None):\n        """Exact closed-form inverse W^{-1}(y) and log|det d(W^{-1})/dy|."""\n        K, B = self.K, self.B\n        widths, heights, cumwidths, cumheights, derivatives = self._compute_knots(raw_params, gate)\n\n        y_c = y.clamp(-B + 1e-6, B - 1e-6)\n        bin_idx = ((y_c.unsqueeze(-1) >= cumheights[..., :-1]).sum(dim=-1, keepdim=True) - 1).clamp(0, K - 1)\n\n        xk = cumwidths.gather(-1, bin_idx).squeeze(-1)\n        wk = widths.gather(-1, bin_idx).squeeze(-1)\n        yk = cumheights.gather(-1, bin_idx).squeeze(-1)\n        hk = heights.gather(-1, bin_idx).squeeze(-1)\n        dk = derivatives.gather(-1, bin_idx).squeeze(-1)\n        dk1 = derivatives.gather(-1, bin_idx + 1).squeeze(-1)\n\n        sk = hk / wk.clamp(min=1e-8)\n        delta_d = dk1 + dk - 2.0 * sk\n        dy = (y_c - yk).clamp(min=0.0)\n\n        a = hk * (sk - dk) + dy * delta_d\n        b = hk * dk - dy * delta_d\n        c = -dy * sk\n        disc = (b.pow(2) - 4.0 * a * c).clamp(min=1e-12)\n        denom = (-b - torch.sqrt(disc)).clamp(max=-1e-12)\n        theta = (2.0 * c / denom).clamp(0.0, 1.0)\n        x_pre = xk + theta * wk\n\n        theta_1m = theta * (1.0 - theta)\n        den = (sk + delta_d * theta_1m).clamp(min=1e-8)\n        grad_num = sk.pow(2) * (dk1 * theta.pow(2) + 2.0 * sk * theta_1m + dk * (1.0 - theta).pow(2))\n        dydx = (grad_num / den.pow(2).clamp(min=1e-12)).clamp(min=1e-6)\n        log_det_inv = -torch.log(dydx).sum(dim=-1)\n\n        return x_pre, log_det_inv\n\n\nclass DeepSetsWarpResiPActor(nn.Module):\n    """Stochastic DeepSets Spline-Warp Actor with Physics-Uncertainty Gate.\n\n    Models pi(y | s) by sampling x_pre ~ N(x_base, sigma(s)^2) and warping\n    through bounded RationalQuadraticWarp1D gated by RLS model_error.\n    """\n    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM,\n                 hidden=128, action_dim=AHU_ACT_DIM, num_bins=8,\n                 min_std=0.02, max_std=0.18):\n        super().__init__()\n        self.action_dim = action_dim\n        self.min_std = float(min_std)\n        self.max_std = float(max_std)\n        self.warp = RationalQuadraticWarp1D(num_bins=num_bins)\n        n_spline = self.warp.num_params_per_dim\n\n        self.zone_enc = nn.Sequential(\n            nn.Linear(zone_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),\n            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),\n        )\n        ctx_dim = hidden * 2 + global_dim + action_dim\n        self.trunk = nn.Sequential(\n            nn.Linear(ctx_dim, 256), nn.LayerNorm(256), nn.GELU(),\n            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),\n        )\n        self.spline_head = nn.Linear(256, action_dim * n_spline)\n        self.gate_head = nn.Linear(256, action_dim)\n        self.std_head = nn.Linear(256, action_dim)\n\n        nn.init.zeros_(self.spline_head.weight)\n        nn.init.zeros_(self.spline_head.bias)\n        nn.init.zeros_(self.gate_head.weight)\n        nn.init.constant_(self.gate_head.bias, -2.0)\n        nn.init.zeros_(self.std_head.weight)\n        nn.init.constant_(self.std_head.bias, -1.2)  # initial std ~ 0.057\n\n    def _encode(self, z_feats, z_mask, g_feats, x_base):\n        h = self.zone_enc(z_feats) * z_mask\n        denom = z_mask.sum(dim=1).clamp(min=1.0)\n        z_mean = h.sum(dim=1) / denom\n        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values\n        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))\n\n        # Physics-Uncertainty Gate Modulation from causal RLS model_error (col 5)\n        me_mean = (z_feats[:, :, 5:6] * z_mask).sum(dim=1) / denom\n        unc_scale = torch.clamp((me_mean - 0.04) / 0.16, min=0.20, max=1.0)\n\n        ctx = torch.cat([z_mean, z_max, g_feats, x_base], dim=-1)\n        feat = self.trunk(ctx)\n\n        raw_spline = self.spline_head(feat).view(-1, self.action_dim, self.warp.num_params_per_dim)\n        gate = unc_scale * torch.sigmoid(self.gate_head(feat))\n        std = self.min_std + (self.max_std - self.min_std) * torch.sigmoid(self.std_head(feat))\n        return raw_spline, gate, std\n\n    def forward(self, z_feats, z_mask, g_feats, x_base, n_antithetic=4):\n        """Antithetic MC expectation over warped stochastic distribution.\n\n        At identity init (W == Identity), E[W(x_base +- eps)] == x_base identically!\n        """\n        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)\n        if n_antithetic <= 0:\n            x_warped, _, log_det = self.warp(x_base, raw_spline, gate=gate.unsqueeze(-1))\n            return x_warped, gate, log_det, std\n\n        # Symmetric quadrature nodes over N(0, std(s)^2) capped inside [-B, B]\n        B_sz = x_base.shape[0]\n        nodes = torch.tensor([0.3, 0.8, 1.5], device=x_base.device, dtype=x_base.dtype).view(1, 3, 1)\n        headroom = (self.warp.B - 1e-4 - x_base.abs()).clamp(min=0.0).unsqueeze(1)  # [B, 1, D]\n        deltas = torch.minimum(std.unsqueeze(1) * nodes, headroom)                   # [B, 3, D]\n        x_pre = torch.cat([x_base.unsqueeze(1) + deltas, x_base.unsqueeze(1) - deltas], dim=1)  # [B, 6, D]\n        rs_exp = raw_spline.unsqueeze(1).expand(B_sz, 6, -1, -1)\n        gt_exp = gate.unsqueeze(1).unsqueeze(-1).expand(B_sz, 6, -1, -1)\n\n        _, y_raw_mc, log_det_mc = self.warp(x_pre, rs_exp, gate=gt_exp)\n        x_expected = y_raw_mc.mean(dim=1).clamp(-1.0, 1.0)\n        log_det_mean = log_det_mc.mean(dim=1)\n        return x_expected, gate, log_det_mean, std\n\n    def sample(self, z_feats, z_mask, g_feats, x_base):\n        """Stochastic sample and exact change-of-variables log_prob."""\n        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)\n        eps = torch.randn_like(x_base)\n        x_pre = x_base + std * eps\n        y_clamped, y_raw, log_det_j = self.warp(x_pre, raw_spline, gate=gate.unsqueeze(-1))\n        log_p0 = (-0.5 * (eps.pow(2) + 2.0 * torch.log(std) + math.log(2.0 * math.pi))).sum(dim=-1)\n        log_prob = log_p0 - log_det_j\n        return y_clamped, log_prob, gate, log_det_j, std\n\n    def log_prob(self, z_feats, z_mask, g_feats, x_base, y_data):\n        """Exact log pi(y_data | s) via closed-form spline inverse for IQL-AWR."""\n        raw_spline, gate, std = self._encode(z_feats, z_mask, g_feats, x_base)\n        x_pre, log_det_inv = self.warp.inverse(y_data, raw_spline, gate=gate.unsqueeze(-1))\n        z_norm = (x_pre - x_base) / std\n        log_p0 = (-0.5 * (z_norm.pow(2) + 2.0 * torch.log(std) + math.log(2.0 * math.pi))).sum(dim=-1)\n        log_pi = log_p0 + log_det_inv\n        return log_pi, gate, log_det_inv, std\n\n\nclass DeepSetsValueNet(nn.Module):\n    """State Value Network V(s) for Implicit Q-Learning (Expectile Regression)."""\n    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM, hidden=128):\n        super().__init__()\n        self.zone_enc = nn.Sequential(\n            nn.Linear(zone_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),\n            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),\n        )\n        self.v_net = nn.Sequential(\n            nn.Linear(hidden * 2 + global_dim, 256), nn.LayerNorm(256), nn.GELU(),\n            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),\n            nn.Linear(256, 1),\n        )\n\n    def forward(self, z_feats, z_mask, g_feats):\n        h = self.zone_enc(z_feats) * z_mask\n        denom = z_mask.sum(dim=1).clamp(min=1.0)\n        z_mean = h.sum(dim=1) / denom\n        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values\n        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))\n        return self.v_net(torch.cat([z_mean, z_max, g_feats], dim=-1))\n\n\nclass _WarpCriticHead(nn.Module):\n    def __init__(self, zone_dim, global_dim, hidden, action_dim):\n        super().__init__()\n        self.zone_enc = nn.Sequential(\n            nn.Linear(zone_dim + action_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),\n            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),\n        )\n        self.q_net = nn.Sequential(\n            nn.Linear(hidden * 2 + global_dim + action_dim, 256), nn.LayerNorm(256), nn.GELU(),\n            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),\n            nn.Linear(256, 1),\n        )\n\n    def forward(self, z_feats, z_mask, g_feats, x_eff):\n        x_exp = x_eff.unsqueeze(1).expand(-1, z_feats.shape[1], -1)\n        h = self.zone_enc(torch.cat([z_feats, x_exp], dim=-1)) * z_mask\n        denom = z_mask.sum(dim=1).clamp(min=1.0)\n        z_mean = h.sum(dim=1) / denom\n        z_max = h.masked_fill(z_mask.expand_as(h) < 0.5, -1e9).max(dim=1).values\n        z_max = torch.where(denom > 0, z_max, torch.zeros_like(z_max))\n        return self.q_net(torch.cat([z_mean, z_max, g_feats, x_eff], dim=-1))\n\n\nclass DeepSetsAHUTwinCritic(nn.Module):\n    """Building-level Twin Critic receiving 4D warped action x_eff in [-1, 1]^4."""\n    def __init__(self, zone_dim=ZONE_FEAT_DIM, global_dim=GLOBAL_FEAT_DIM,\n                 hidden=128, action_dim=AHU_ACT_DIM):\n        super().__init__()\n        self.q1 = _WarpCriticHead(zone_dim, global_dim, hidden, action_dim)\n        self.q2 = _WarpCriticHead(zone_dim, global_dim, hidden, action_dim)\n\n    def forward(self, z_feats, z_mask, g_feats, x_eff):\n        return self.q1(z_feats, z_mask, g_feats, x_eff), self.q2(z_feats, z_mask, g_feats, x_eff)\n\n    def Q1(self, z_feats, z_mask, g_feats, x_eff):\n        return self.q1(z_feats, z_mask, g_feats, x_eff)\n\n\ndef extract_warp_state(policy, obs_arr, temperature, low, high, mode, oat_ema):\n    outside_ahu = np.array([\n        policy._temperature(obs_arr, idx, \'outside_air_temperature_sensor\', 290.0)\n        for idx in policy.outside_obs\n    ])\n    outside = outside_ahu[policy.zone_ahu]\n    mean_oat = float(np.mean(outside_ahu))\n    oat_ema = mean_oat if oat_ema is None else (0.98 * oat_ema + 0.02 * mean_oat)\n\n    free = policy._free_drift(temperature, outside, mode)\n    z_feats = np.column_stack([\n        np.clip((temperature - low) / 2.0, -5.0, 5.0),\n        np.clip((high - temperature) / 2.0, -5.0, 5.0),\n        np.clip(free / 2.0, -3.0, 3.0),\n        np.clip((outside - temperature) / 20.0, -5.0, 5.0),\n        np.clip((policy.slow_temperature - temperature) / 5.0, -5.0, 5.0),\n        np.clip(policy.model_error, 0.0, 2.0),\n        np.clip(policy.residual / 2.0, -2.0, 2.0),\n        np.log(np.clip(policy.theta[:, 5], 1e-3, 10.0)),\n        np.log1p(policy.zone_area) / 5.0,\n    ]).astype(np.float32)\n    z_feats = np.nan_to_num(z_feats, nan=0.0, posinf=5.0, neginf=-5.0)\n\n    hour = policy._hour(obs_arr)\n    sine_idx = policy.field_index.get(\'hod_sin_000\')\n    cos_idx = policy.field_index.get(\'hod_cos_000\')\n    hod_sin = float(obs_arr[sine_idx]) if sine_idx is not None else math.sin(hour * math.pi / 12.0)\n    hod_cos = float(obs_arr[cos_idx]) if cos_idx is not None else math.cos(hour * math.pi / 12.0)\n\n    c_now = float(obs_arr[policy.comfort_now_index] > 0.5) if policy.comfort_now_index is not None else float(mode)\n    c_soon = float(obs_arr[policy.comfort_soon_index] > 0.5) if policy.comfort_soon_index is not None else c_now\n    hours_to_start = ((policy.start_hour - hour) % 24.0) / 24.0\n    occ_raw = float(obs_arr[policy.occupancy_index]) if policy.occupancy_index is not None else 0.0\n    occ_norm = math.log1p(max(occ_raw, 0.0)) / 5.0 if np.isfinite(occ_raw) else 0.0\n    water_obs = policy._temperature(obs_arr, policy.water_sensor, \'supply_water_temperature_sensor\', 340.0)\n    heat_recency = math.exp(-max(0, policy.step_count - policy.last_heating_step) / 144.0)\n\n    g_feats = np.array([\n        hod_sin, hod_cos,\n        np.clip((mean_oat - 290.0) / 15.0, -3.0, 3.0),\n        np.clip((mean_oat - oat_ema) / 5.0, -3.0, 3.0),\n        c_now, c_soon, hours_to_start, occ_norm,\n        np.clip((water_obs - 330.0) / 20.0, -3.0, 3.0),\n        heat_recency,\n    ], dtype=np.float32)\n    g_feats = np.nan_to_num(g_feats, nan=0.0, posinf=3.0, neginf=-3.0)\n    return z_feats, g_feats, oat_ema, outside_ahu, outside, free\n\n\nclass WarpResiPMixin:\n    """Mixin layering Stochastic Warp RL + Physics-Uncertainty Gate + Physics Safety Shield."""\n    def __init__(self, building_meta_data, action_spec=None, parameters=None,\n                 actors=None, action_modifier=None, dev=None, smooth_alpha=0.5,\n                 comfort_shield=True, target_water_only=False):\n        self._actors = actors if isinstance(actors, (list, tuple)) else ([actors] if actors is not None else [])\n        self._action_modifier = action_modifier\n        self._dev = dev or torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self._smooth_alpha = float(smooth_alpha)\n        self._comfort_shield = bool(comfort_shield)\n        self._target_water_only = bool(target_water_only)\n        super().__init__(building_meta_data, action_spec, parameters)\n\n    def reset(self):\n        super().reset()\n        self._oat_ema = None\n        self._last_transition = None\n        self._last_gate = np.zeros(AHU_ACT_DIM, dtype=np.float32)\n        self._last_thermal_sample = None\n        self._prev_delta_x = np.zeros(AHU_ACT_DIM, dtype=np.float32)\n        self._shield_checks = 0\n        self._shield_hits = 0\n\n    def _learn(self, observed):\n        delay = int(self.parameters[\'observation_delay\'])\n        self._last_thermal_sample = None\n        if self.previous_observed is not None and len(self.history) > delay:\n            applied = list(self.history)[-(delay + 1)]\n            old = self.previous_observed\n            features = self._features(old, applied[\'outside\'], applied[\'mode\'], applied[\'slow\'])\n            drift = np.einsum(\'ij,ij->i\', self.theta[:, :5], features) + self.residual\n            gain = self.theta[:, 5] / 1000.0\n            pred_next = (old + drift + gain * (applied[\'conductance\'] * applied[\'supply\'] + applied[\'reheat\'])) / (\n                1.0 + gain * applied[\'conductance\'])\n            power = (applied[\'conductance\'] * (applied[\'supply\'] - observed) + applied[\'reheat\']) / 1000.0\n            self._last_thermal_sample = {\n                \'old\': old.copy(),\n                \'pred_next\': pred_next.copy(),\n                \'observed\': observed.copy(),\n                \'features\': features.copy(),\n                \'power\': power.copy(),\n                \'model_error\': self.model_error.copy(),\n                \'residual\': self.residual.copy(),\n                \'theta_gain\': self.theta[:, 5].copy(),\n                \'zone_area\': self.zone_area.copy(),\n            }\n        super()._learn(observed)\n\n    def _ahu_step_metrics(self, group, temperature, free, low, high, outside, choice):\n        t, d, lo, hi = temperature[group], free[group], low[group], high[group]\n        response = self.theta[group, 5]\n        inside = float(np.average(t, weights=self.zone_area[group]))\n        supply = choice[\'supply\']\n        conductance = choice[\'conductance\']\n        reheat = choice[\'reheat\']\n        predicted = (t + d + response * (conductance * supply + reheat) / 1000.0) / (\n            1.0 + response * conductance / 1000.0)\n        violation = np.maximum(lo - predicted, 0.0) + np.maximum(predicted - hi, 0.0)\n        thermal_penalty = float(np.sum(violation + 2.0 * violation ** 2))\n        if choice[\'run\'] == 0:\n            obj = float(self.parameters[\'comfort_cost\']) * thermal_penalty\n        else:\n            flow = conductance / self.AIR_CAPACITY\n            mixed = choice[\'recirc\'] * inside + (1.0 - choice[\'recirc\']) * outside\n            fan = float(flow.sum()) * choice[\'pressure\'] * (2.0 - choice[\'recirc\']) / 0.9\n            chiller = float((flow * self.AIR_CAPACITY * np.maximum(mixed - supply, 0.0)).sum()) / 3.5\n            gas = float(reheat.sum())\n            obj = fan + chiller + float(self.parameters[\'gas_cost_ratio\']) * gas + float(self.parameters[\'comfort_cost\']) * thermal_penalty\n        return thermal_penalty, float(obj)\n\n    def _reproject_ahu(self, group, temperature, free, low, high, max_flow, outside, water,\n                       setpoint_w, pressure_w, recirc_w):\n        t, d, lo, hi = temperature[group], free[group], low[group], high[group]\n        response = self.theta[group, 5]\n        capacity = max_flow[group]\n        inside = float(np.average(t, weights=self.zone_area[group]))\n\n        mixed = recirc_w * inside + (1.0 - recirc_w) * outside\n        supply = min(mixed, setpoint_w)\n\n        cool_need = np.maximum(t + d - hi, 0.0) * 1000.0 / response\n        heat_need = np.maximum(lo - t - d, 0.0) * 1000.0 / response\n        cooling_flow = cool_need / (self.AIR_CAPACITY * np.maximum(hi - supply, 0.05))\n        heating_flow = heat_need / (self.AIR_CAPACITY * np.maximum(self.MAX_DISCHARGE_K - lo, 0.5))\n\n        maximum_water_heat = self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * max(water - supply, 0.0)\n        maximum_heating_flow = maximum_water_heat / (self.AIR_CAPACITY * max(self.MAX_DISCHARGE_K - supply, 0.1))\n        heating_flow = np.minimum(heating_flow, maximum_heating_flow)\n        needed_flow = np.maximum(cooling_flow, heating_flow)\n\n        pressure_ratio = math.sqrt(pressure_w / self.DESIGN_PRESSURE)\n        dampers = np.clip(needed_flow / (capacity * pressure_ratio), self.MIN_DAMPER, 1.0)\n        flow = dampers * capacity * pressure_ratio\n        conductance = flow * self.AIR_CAPACITY\n\n        water_gap = max(water - supply, 0.0)\n        wanted_reheat = np.maximum(\n            (lo - t - d) * 1000.0 / response + conductance * (lo - supply), 0.0)\n        reheat_limit = np.minimum(\n            conductance * max(self.MAX_DISCHARGE_K - supply, 0.0),\n            self.WATER_VALVE_CAPACITY * math.sqrt(20.0) * water_gap)\n        reheat = np.minimum(wanted_reheat, reheat_limit)\n\n        return {\n            \'run\': 1, \'pressure\': float(pressure_w), \'setpoint\': float(setpoint_w),\n            \'recirc\': float(recirc_w), \'damper\': dampers, \'conductance\': conductance,\n            \'supply\': np.full(group.size, supply), \'reheat\': reheat,\n        }\n\n    def get_action(self, obs):\n        obs_arr = np.asarray(obs, dtype=float).reshape(-1)\n        observed = self._zone_temperatures(obs_arr)\n        if self.slow_temperature is None:\n            self.slow_temperature = observed.copy()\n        self._learn(observed)\n\n        outside_ahu = np.array([\n            self._temperature(obs_arr, index, \'outside_air_temperature_sensor\', 290.0)\n            for index in self.outside_obs\n        ])\n        outside = outside_ahu[self.zone_ahu]\n        mode = bool(obs_arr[self.comfort_now_index] > 0.5) if self.comfort_now_index is not None else True\n\n        temperature = observed.copy()\n        if self.history and int(self.parameters[\'observation_delay\']) == 1:\n            last = self.history[-1]\n            drift = self._free_drift(observed, last[\'outside\'], last[\'mode\'])\n            gain = self.theta[:, 5] / 1000.0\n            temperature = (observed + drift + gain * (last[\'conductance\'] * last[\'supply\'] + last[\'reheat\'])) / (\n                1.0 + gain * last[\'conductance\'])\n            temperature = np.clip(temperature, observed - 8.0, observed + 8.0)\n\n        low, high, mode = self._band(obs_arr, temperature)\n        z_f, g_f, self._oat_ema, _, _, free = extract_warp_state(\n            self, obs_arr, temperature, low, high, mode, self._oat_ema)\n\n        max_flow = np.full(self.n_zones, 2.0)\n        valid = self.zone_flow_obs >= 0\n        if np.any(valid):\n            flow_reading = obs_arr[self.zone_flow_obs[valid]]\n            max_flow[valid] = np.where(np.isfinite(flow_reading) & (flow_reading > 0.0), flow_reading, 2.0)\n        max_flow = np.clip(max_flow, 0.01, 100.0)\n\n        water_observed = self._temperature(obs_arr, self.water_sensor, \'supply_water_temperature_sensor\', 340.0)\n        if np.any(temperature + free < low):\n            self.last_heating_step = self.step_count\n        recent_heat = self.step_count - self.last_heating_step < 24\n        water_available = water_observed\n\n        base_choices = []\n        any_reheat_10 = False\n        for i, group in enumerate(self.groups):\n            if not group.size:\n                base_choices.append(None)\n                continue\n            c = self._choose_ahu(group, temperature, free, low, high, max_flow,\n                                 float(outside_ahu[i]), float(water_available))\n            base_choices.append(c)\n            if np.any(c[\'reheat\'] > 10.0):\n                any_reheat_10 = True\n\n        if any_reheat_10:\n            self.last_heating_step = self.step_count\n        hold_steps = getattr(self, \'HOLD_STEPS\', 24)\n        hold_active = self.step_count - self.last_heating_step < hold_steps\n        base_water_target = float(self.parameters[\'hot_water_target\']) if (recent_heat or any_reheat_10 or hold_active) else 310.0\n\n        active_indices = [i for i, c in enumerate(base_choices) if c is not None and c[\'run\'] == 1]\n        if active_indices:\n            weights = np.array([self.zone_area[self.groups[i]].sum() for i in active_indices], dtype=float)\n            weights /= max(weights.sum(), 1e-8)\n            sat_b = sum(w * base_choices[i][\'setpoint\'] for w, i in zip(weights, active_indices))\n            press_b = sum(w * base_choices[i][\'pressure\'] for w, i in zip(weights, active_indices))\n            recirc_b = sum(w * base_choices[i][\'recirc\'] for w, i in zip(weights, active_indices))\n        else:\n            sat_b, press_b, recirc_b = 305.0, 0.0, 0.7\n\n        x_base = np.array([\n            np.clip((sat_b - 295.0) / 10.0, -1.0, 1.0),\n            np.clip((press_b - 1000.0) / 1000.0, -1.0, 1.0),\n            np.clip((recirc_b - 0.5) / 0.5, -1.0, 1.0),\n            np.clip((base_water_target - 330.0) / 20.0, -1.0, 1.0),\n        ], dtype=np.float32)\n\n        gate_val = np.zeros(AHU_ACT_DIM, dtype=np.float32)\n        if self._action_modifier is not None:\n            x_eff = self._action_modifier(self.step_count, z_f, g_f, x_base)\n        elif self._actors:\n            zt = torch.tensor(z_f, dtype=torch.float32, device=self._dev).unsqueeze(0)\n            zm = torch.ones((1, z_f.shape[0], 1), dtype=torch.float32, device=self._dev)\n            gt = torch.tensor(g_f, dtype=torch.float32, device=self._dev).unsqueeze(0)\n            xb = torch.tensor(x_base, dtype=torch.float32, device=self._dev).unsqueeze(0)\n            with torch.no_grad():\n                outs = [net(zt, zm, gt, xb) for net in self._actors]\n            x_raw = np.mean([o[0].squeeze(0).cpu().numpy() for o in outs], axis=0).astype(np.float32)\n            gate_val = np.mean([o[1].squeeze(0).cpu().numpy() for o in outs], axis=0).astype(np.float32)\n            # Temporal delta smoothing (prevents 5-minute actuator hunting)\n            raw_dx = x_raw - x_base\n            if not np.allclose(raw_dx, 0.0, atol=1e-7):\n                sm_dx = self._smooth_alpha * raw_dx + (1.0 - self._smooth_alpha) * self._prev_delta_x\n                self._prev_delta_x = sm_dx\n                x_eff = np.clip(x_base + sm_dx, -1.0, 1.0).astype(np.float32)\n            else:\n                self._prev_delta_x = np.zeros(AHU_ACT_DIM, dtype=np.float32)\n                x_eff = x_base.copy()\n        else:\n            x_eff = x_base.copy()\n\n        if self._target_water_only:\n            x_eff = np.array([x_base[0], x_base[1], x_base[2], x_eff[3]], dtype=np.float32)\n\n        delta_x = x_eff - x_base\n        if np.allclose(delta_x, 0.0, atol=1e-7):\n            water_target = base_water_target\n        else:\n            water_target = float(np.clip(330.0 + 20.0 * float(x_eff[3]), 310.0, 350.0))\n            if self._comfort_shield and self._action_modifier is None and (any_reheat_10 or np.any(temperature + free < low)):\n                water_target = max(water_target, base_water_target)\n\n        continuous = np.full(self.n_continuous, -1.0, dtype=np.float32)\n        discrete = np.zeros(self.n_discrete, dtype=np.int32)\n        conductance = np.zeros(self.n_zones)\n        supply = outside.copy()\n        reheat = np.zeros(self.n_zones)\n\n        for i, group in enumerate(self.groups):\n            c = base_choices[i]\n            if c is None:\n                continue\n            if c[\'run\'] == 1 and not self._target_water_only and not np.allclose(delta_x[:3], 0.0, atol=1e-7):\n                sat_w = float(np.clip(c[\'setpoint\'] + 10.0 * float(delta_x[0]), 285.0, 305.0))\n                press_w = float(np.clip(c[\'pressure\'] + 1000.0 * float(delta_x[1]), self.MIN_PRESSURE, self.DESIGN_PRESSURE))\n                recirc_w = float(np.clip(c[\'recirc\'] + 0.5 * float(delta_x[2]), 0.0, self.MAX_RECIRCULATION))\n                c_w = self._reproject_ahu(\n                    group, temperature, free, low, high, max_flow,\n                    float(outside_ahu[i]), float(water_available), sat_w, press_w, recirc_w)\n                if self._comfort_shield and self._action_modifier is None:\n                    self._shield_checks += 1\n                    pen_b, _ = self._ahu_step_metrics(group, temperature, free, low, high, float(outside_ahu[i]), c)\n                    pen_w, obj_w = self._ahu_step_metrics(group, temperature, free, low, high, float(outside_ahu[i]), c_w)\n                    if (not np.isfinite(obj_w)) or (pen_w > pen_b + 1e-6):\n                        self._shield_hits += 1\n                    else:\n                        c = c_w\n                else:\n                    c = c_w\n\n            discrete[i] = c[\'run\']\n            continuous[self.temp_indices[i]] = TransferPolicy._native_action(\n                self, f\'{self.ahu_ids[i]}_supply_air_temperature_setpoint\', c[\'setpoint\'], 285.0, 305.0)\n            continuous[self.pressure_indices[i]] = TransferPolicy._native_action(\n                self, f\'{self.ahu_ids[i]}_supply_air_static_pressure_setpoint\', c[\'pressure\'], 0.0, 2000.0)\n            continuous[self.recirculation_indices[i]] = TransferPolicy._native_action(\n                self, f\'{self.ahu_ids[i]}_recirculation_percentage_command\', c[\'recirc\'], 0.0, 1.0)\n            continuous[self.damper_indices[group]] = TransferPolicy._native_action(\n                self, \'supply_air_damper_percentage_command\', c[\'damper\'], 0.0, 1.0)\n            conductance[group] = c[\'conductance\']\n            supply[group] = c[\'supply\']\n            reheat[group] = c[\'reheat\']\n\n        if np.any(reheat > 10.0):\n            self.last_heating_step = self.step_count\n\n        valve_need = reheat / (self.WATER_VALVE_CAPACITY * np.maximum(water_available - supply, 1.0))\n        pressure_water = float(np.clip(np.max(valve_need) ** 2 * 1.05, 0.0, 20.0))\n        if pressure_water > 1e-8:\n            valves = np.clip(valve_need / math.sqrt(pressure_water), 0.0, 1.0)\n            discrete[-1] = 1\n        else:\n            valves = np.zeros(self.n_zones)\n\n        continuous[self.valve_indices] = TransferPolicy._native_action(\n            self, \'reheat_valve_percentage_command\', valves, 0.0, 1.0)\n        if self.water_indices:\n            continuous[self.water_indices[0]] = TransferPolicy._native_action(\n                self, \'supply_water_temperature_setpoint\', water_target, 310.0, 350.0)\n        if len(self.pressure_indices) > self.num_ahus:\n            continuous[self.pressure_indices[-1]] = TransferPolicy._native_action(\n                self, \'differential_pressure_setpoint\', pressure_water, 0.0, 20.0)\n\n        actual_reheat = np.minimum(\n            self.WATER_VALVE_CAPACITY * math.sqrt(pressure_water) * valves * np.maximum(water_available - supply, 0.0),\n            conductance * np.maximum(self.MAX_DISCHARGE_K - supply, 0.0))\n        self.history.append({\'conductance\': conductance, \'supply\': supply,\n                             \'reheat\': actual_reheat, \'outside\': outside.copy(),\n                             \'mode\': mode, \'slow\': self.slow_temperature.copy()})\n        self.slow_temperature += (temperature - self.slow_temperature) / 48.0\n        self.previous_observed = observed.copy()\n        self.step_count += 1\n        self._last_transition = (z_f, g_f, x_base, x_eff)\n        self._last_gate = gate_val\n\n        action = {\'continuous_action\': np.clip(np.nan_to_num(continuous, nan=-1.0), -1.0, 1.0).astype(np.float32),\n                  \'discrete_action\': discrete.astype(np.int32)}\n        self.last_action = action\n        return action\n\n\ndef make_warp_policy(base_cls):\n    class _ConfiguredWarpResiPPolicy(WarpResiPMixin, base_cls):\n        name = f\'warp_resip_{getattr(base_cls, "name", "base")}\'\n    return _ConfiguredWarpResiPPolicy\n\n\nclass WarpResiPPolicy(WarpResiPMixin, EmptyDayConsistentPolicy):\n    name = \'warp_resip_empty_day\'\n\n\nclass WarpResiPCausalPolicy(WarpResiPMixin, CausalIdentifiedReservePolicy):\n    name = \'warp_resip_causal_identified\'\n\n\ndef verify_warp_resip_module():\n    """Sanity check: identity init, closed-form invertibility, log_prob & gradients."""\n    dev = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    actor = DeepSetsWarpResiPActor().to(dev)\n    critic = DeepSetsAHUTwinCritic().to(dev)\n    vnet = DeepSetsValueNet().to(dev)\n\n    z = torch.randn(4, 12, ZONE_FEAT_DIM, device=dev)\n    z[:, :, 5] = 0.25  # realistic RLS model_error\n    m = torch.ones(4, 12, 1, device=dev)\n    g = torch.randn(4, GLOBAL_FEAT_DIM, device=dev)\n    xb = torch.linspace(-0.9, 0.9, 16, device=dev).view(4, 4)\n\n    # 1. Exact Identity check with Antithetic MC\n    xw, gate, log_det, std = actor(z, m, g, xb, n_antithetic=4)\n    max_id_err = float((xw - xb).abs().max().item())\n    max_ld_err = float(log_det.abs().max().item())\n\n    # 2. Exact closed-form inverse check on nontrivial random spline params\n    raw_test = torch.randn(4, 4, actor.warp.num_params_per_dim, device=dev) * 0.8\n    _, y_raw, ld_fwd = actor.warp(xb, raw_test)\n    x_rec, ld_inv = actor.warp.inverse(y_raw, raw_test)\n    max_inv_err = float((x_rec - xb).abs().max().item())\n    max_jac_err = float((ld_fwd + ld_inv).abs().max().item())\n\n    # 3. Gradient flow check through IQL log_prob + CAPS\n    log_pi, _, _, _ = actor.log_prob(z, m, g, xb, xb + 0.05)\n    v_val = vnet(z, m, g)\n    q1, _ = critic(z, m, g, xw)\n    loss = -(log_pi.mean()) + (v_val - q1.detach()).pow(2).mean()\n    loss.backward()\n    has_grad = all(p.grad is not None and torch.isfinite(p.grad).all()\n                   for p in actor.parameters() if p.requires_grad)\n    return {\n        \'max_identity_err\': max_id_err,\n        \'max_logdet_err\': max_ld_err,\n        \'max_inv_err\': max_inv_err,\n        \'max_jac_err\': max_jac_err,\n        \'grad_ok\': has_grad,\n    }\n')
    (Path('/kaggle/working') / 'collect_data_v2.py').write_text('"""SOTA Data Collection & Quality Audit Engine (v2) — Track Data Only (Model-Neutral).\n\nAddresses 6 empirical/SOTA gaps against Wave 0 (864s single-policy):\n1. Multi-Policy Coverage (arXiv:2408.07986): anchor EmptyDay + Orthogonal Explore + PIDWaterReset.\n2. Orthogonal Single-Axis Excitation: decouples SAT/Press/Recirc/Water (low VIF / collinearity).\n3. POMDP History & Geometry Enrichment: ep_id, step_idx, dow_sin/cos, zone_ua, dT_30m, dOAT_30m, ahu_spread.\n4. Full-Week Temporal Realism (default STEPS=2016 = 7 days vs 864 = 3 days from Sun 06:00).\n5. BOPTEST KPI Decomposition: separate Comfort Regret, Energy Cost, and Carbon Emission per step.\n6. Automated SOTA Data Audit: Temporal drift (0..864 vs 864..2016), Regret Ratio, Violation %, Excitation VIF.\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom tqdm.auto import tqdm\n\n_HERE = str(Path(__file__).resolve().parent)\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\n\nfrom physics_foundation import EmptyDayConsistentPolicy\nfrom warp_resip import AHU_ACT_DIM, ZONE_FEAT_DIM, WarpResiPPolicy\n\n\ndef _step_kpis(env):\n    """Read BOPTEST-style KPI decomposition without mutating simulator accumulators."""\n    try:\n        rr = env.reward_function.compute_reward(env.building.reward_info)\n        return (\n            float(rr.normalized_productivity_regret),\n            float(rr.normalized_energy_cost),\n            float(rr.normalized_carbon_emission),\n        )\n    except Exception:\n        return 0.0, 0.0, 0.0\n\n\ndef _extract_dow(policy, obs_arr):\n    s_idx = policy.field_index.get(\'dow_sin_000\')\n    c_idx = policy.field_index.get(\'dow_cos_000\')\n    if s_idx is not None and c_idx is not None:\n        return float(obs_arr[s_idx]), float(obs_arr[c_idx])\n    # Fallback cyclic week (period=7*24h=168h, step_count in 5-min ticks)\n    ang = 2.0 * math.pi * ((policy.step_count / 12.0) % 168.0) / 168.0\n    return math.sin(ang), math.cos(ang)\n\n\ndef _extract_4d_from_action(policy, act, fallback_xb):\n    """Project any policy\'s hybrid action dict into normalized 4D [SAT, Press, Recirc, Water]."""\n    cont = np.asarray(act[\'continuous_action\'], dtype=np.float32)\n    disc = np.asarray(act[\'discrete_action\'], dtype=np.int32)\n    active = [i for i in range(policy.num_ahus) if disc[i] == 1]\n    if active:\n        weights = np.array([policy.zone_area[policy.groups[i]].sum() for i in active], dtype=np.float64)\n        weights /= max(weights.sum(), 1e-8)\n        sat_n = float(sum(w * cont[policy.temp_indices[i]] for w, i in zip(weights, active)))\n        prs_n = float(sum(w * cont[policy.pressure_indices[i]] for w, i in zip(weights, active)))\n        rec_n = float(sum(w * cont[policy.recirculation_indices[i]] for w, i in zip(weights, active)))\n    else:\n        sat_n, prs_n, rec_n = float(fallback_xb[0]), float(fallback_xb[1]), float(fallback_xb[2])\n    wat_n = float(cont[policy.water_indices[0]]) if policy.water_indices else float(fallback_xb[3])\n    return np.clip(np.array([sat_n, prs_n, rec_n, wat_n], dtype=np.float32), -1.0, 1.0)\n\n\ndef _ortho_modifier_factory(block_steps=6, rng=None):\n    """Orthogonal single-axis perturbation so excitation VIF stays near 1.0."""\n    rng = rng or np.random.default_rng(42)\n    state = {\'axis\': 0, \'delta\': 0.0, \'tag\': \'ortho_sat\'}\n    axes = [\'ortho_sat\', \'ortho_press\', \'ortho_recirc\', \'ortho_water\']\n\n    def _mod(step_cnt, z_f, g_f, xb):\n        if step_cnt % block_steps == 0:\n            ax = int(rng.integers(0, 4))\n            state[\'axis\'] = ax\n            state[\'tag\'] = axes[ax]\n            if ax == 0:\n                state[\'delta\'] = float(rng.uniform(-0.25, 0.25))\n            elif ax == 1:\n                state[\'delta\'] = float(rng.uniform(-0.35, 0.25))\n            elif ax == 2:\n                state[\'delta\'] = float(rng.uniform(-0.45, 0.30))\n            else:\n                # Wider downward water exploration during idle/low-heat, gentle during active\n                ahu_on = xb[1] > -0.99\n                state[\'delta\'] = float(rng.uniform(-0.25, 0.05) if ahu_on else rng.uniform(-0.80, 0.05))\n        x = xb.copy()\n        ax = state[\'axis\']\n        if ax < 3 and xb[1] <= -0.99:\n            return x  # AHU off -> keep base AHU values untouched\n        x[ax] = float(np.clip(x[ax] + state[\'delta\'], -1.0, 1.0))\n        return x\n\n    return _mod, state\n\n\ndef _rollout_episode_v2(sc_path, bldg, sc_key, steps, mode, load_env_fn, create_meta_fn,\n                        pid_cls=None, block_steps=6, seed=42):\n    """Run one scenario episode under mode in {\'anchor\', \'ortho_explore\', \'pid_water\'}."""\n    env = load_env_fn(sc_path / \'config.gin\')\n    obs, _ = env.reset()\n    meta = create_meta_fn(env, sc_path)\n    rng = np.random.default_rng(seed)\n\n    if mode == \'anchor\':\n        pol = WarpResiPPolicy(meta, env.action_spec(), actors=[])\n        ctrl_pol = None\n        state_ref = {\'tag\': \'anchor_empty_day\'}\n    elif mode == \'ortho_explore\':\n        mod_fn, state_ref = _ortho_modifier_factory(block_steps=block_steps, rng=rng)\n        pol = WarpResiPPolicy(meta, env.action_spec(), action_modifier=mod_fn)\n        ctrl_pol = None\n    elif mode == \'pid_water\':\n        assert pid_cls is not None\n        pol = WarpResiPPolicy(meta, env.action_spec(), actors=[])\n        ctrl_pol = pid_cls(meta, env.action_spec())\n        state_ref = {\'tag\': \'pid_water\'}\n    else:\n        raise ValueError(f\'Unknown mode: {mode}\')\n\n    zone_ua_norm = (np.log1p(np.clip(pol.zone_ua, 0.0, 1e5)) / 4.0).astype(np.float32)\n\n    z_l, g_l, xb_l, xe_l = [], [], [], []\n    z_ex_l, g_ex_l = [], []\n    r_l, reg_l, en_l, cb_l = [], [], [], []\n    tags_l, th_rows = [], []\n\n    temp_hist, oat_hist = [], []\n    prev_gf, prev_gex, prev_zex = None, None, None\n\n    for st in range(steps):\n        obs_arr = np.asarray(obs, dtype=float).reshape(-1)\n        act_base_ref = pol.get_action(obs_arr)\n        zf, gf, xb, xe = pol._last_transition\n\n        if ctrl_pol is not None:\n            act = ctrl_pol.get_action(obs_arr)\n            xe = _extract_4d_from_action(pol, act, xb)\n        else:\n            act = act_base_ref\n\n        temps = pol._zone_temperatures(obs_arr)\n        oat_c = float(gf[2]) * 15.0 + 290.0 - 273.15\n        temp_hist.append(temps.copy())\n        oat_hist.append(oat_c)\n\n        dT_30m = ((temps - temp_hist[max(0, st - 6)]) / 2.0).astype(np.float32)\n        d_oat_30m = float((oat_c - oat_hist[max(0, st - 6)]) / 5.0)\n        dow_sin, dow_cos = _extract_dow(pol, obs_arr)\n\n        spread = np.zeros(pol.n_zones, dtype=np.float32)\n        for grp in pol.groups:\n            if grp.size > 1:\n                spread[grp] = float(np.max(temps[grp]) - np.min(temps[grp])) / 2.0\n        z_ex = np.column_stack([zone_ua_norm, np.clip(dT_30m, -5.0, 5.0), np.clip(spread, 0.0, 5.0)]).astype(np.float32)\n\n        # Thermal residual capture strictly on physics-consistent EmptyDay/Ortho trajectories\n        if ctrl_pol is None and pol._last_thermal_sample is not None and prev_gf is not None:\n            ts = pol._last_thermal_sample\n            err = ts[\'observed\'] - ts[\'pred_next\']\n            nz = len(err)\n            for j in range(nz):\n                th_rows.append([\n                    bldg, sc_key, state_ref[\'tag\'], st - 1, j,\n                    float(prev_gf[0]), float(prev_gf[1]), float(prev_gf[7]),\n                    float(ts[\'features\'][j, 1]), float(ts[\'features\'][j, 2]),\n                    float(ts[\'features\'][j, 3]), float(ts[\'features\'][j, 4]),\n                    float(ts[\'power\'][j]), float(ts[\'model_error\'][j]), float(ts[\'residual\'][j]),\n                    float(ts[\'theta_gain\'][j]), float(ts[\'zone_area\'][j]),\n                    float(prev_zex[j, 0]), float(prev_zex[j, 1]), float(prev_zex[j, 2]),\n                    float(prev_gex[0]), float(prev_gex[1]), float(prev_gex[2]),\n                    float(ts[\'old\'][j]), float(ts[\'pred_next\'][j]), float(ts[\'observed\'][j]), float(err[j]),\n                ])\n\n        obs_next, rew, _, _, _ = env.step(act)\n        reg, en, cb = _step_kpis(env)\n        viol_flag = float(reg > 1e-5)\n        g_ex = np.array([dow_sin, dow_cos, np.clip(d_oat_30m, -3.0, 3.0), viol_flag], dtype=np.float32)\n\n        z_l.append(zf); g_l.append(gf); xb_l.append(xb); xe_l.append(xe)\n        z_ex_l.append(z_ex); g_ex_l.append(g_ex)\n        r_l.append(float(rew)); reg_l.append(reg); en_l.append(en); cb_l.append(cb)\n        tags_l.append(state_ref[\'tag\'])\n\n        prev_gf, prev_gex, prev_zex = gf, g_ex, z_ex\n        obs = obs_next\n\n    return {\n        \'bldg\': bldg, \'sc_key\': sc_key, \'mode\': mode,\n        \'z_l\': z_l, \'g_l\': g_l, \'xb_l\': xb_l, \'xe_l\': xe_l,\n        \'z_ex_l\': z_ex_l, \'g_ex_l\': g_ex_l,\n        \'r_l\': r_l, \'reg_l\': reg_l, \'en_l\': en_l, \'cb_l\': cb_l,\n        \'tags_l\': tags_l, \'th_rows\': th_rows,\n    }\n\n\ndef _compute_vif(X):\n    """Variance Inflation Factor per column on standardized active delta actions + OAT."""\n    if X.shape[0] < 30:\n        return [np.nan] * X.shape[1]\n    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)\n    vifs = []\n    for j in range(X.shape[1]):\n        y = X[:, j]\n        others = np.delete(X, j, axis=1)\n        others = np.column_stack([others, np.ones(len(y))])\n        coef, _, _, _ = np.linalg.lstsq(others, y, rcond=None)\n        pred = others @ coef\n        ss_res = float(np.sum((y - pred) ** 2))\n        ss_tot = float(np.sum((y - y.mean()) ** 2))\n        r2 = 1.0 - ss_res / max(ss_tot, 1e-12)\n        vifs.append(1.0 / max(1.0 - r2, 1e-6))\n    return vifs\n\n\ndef audit_dataset_v2(BUF, df_audit, df_th, out_dir=None):\n    """Run SOTA Data Quality Audit (Temporal Drift, Regret Ratio, Violation %, Excitation VIF, BOPTEST KPI)."""\n    print(\'=\' * 92)\n    print(\'SOTA DATA QUALITY AUDIT REPORT (V2)\')\n    print(\'=\' * 92)\n\n    # 1. Temporal Realism: 0..864 (3d) vs 864..end (rest of week)\n    if df_audit[\'step\'].max() >= 900:\n        df_audit[\'window_split\'] = np.where(df_audit[\'step\'] < 864, \'Day0_3 (0-864)\', \'Day3_7 (864+)\')\n        t_cmp = df_audit[df_audit[\'policy_tag\'] == \'anchor_empty_day\'].groupby(\'window_split\').agg(\n            steps=(\'reward_1step\', \'count\'),\n            active_pct=(\'is_active\', lambda s: round(float(s.mean() * 100), 2)),\n            occ_pct=(\'occ_norm\', lambda s: round(float((s > 0.01).mean() * 100), 2)),\n            rew_per_step=(\'reward_1step\', lambda s: round(float(s.mean()), 5)),\n            comfort_regret=(\'comfort_regret\', lambda s: round(float(s.mean()), 5)),\n            energy_cost=(\'energy_cost\', lambda s: round(float(s.mean()), 5)),\n            viol_pct=(\'comfort_viol_flag\', lambda s: round(float(s.mean() * 100), 2)),\n        )\n        print(\'\\n1. KIEM DINH PHAN PHOI THOI GIAN (3 NGAY DAU vs CAC NGAY TIEP THEO - ANCHOR):\')\n        print(t_cmp.to_string())\n\n    # 2. Coverage & Regret Ratio (arXiv:2408.07986)\n    print(\'\\n2. REGRET RATIO (delta_tau), TY LE VI PHAM & DO PHU HANH DONG THEO POLICY:\')\n    pol_stats = df_audit.groupby(\'policy_tag\').agg(\n        samples=(\'reward_1step\', \'count\'),\n        active_pct=(\'is_active\', lambda s: round(float(s.mean() * 100), 2)),\n        rew_n_mean=(\'reward_nstep\', lambda s: round(float(s.mean()), 5)),\n        anc_n_mean=(\'reward_anchor\', lambda s: round(float(s.mean()), 5)),\n        win_vs_anchor_pct=(\'regret_vs_anchor\', lambda s: round(float((s > 0).mean() * 100), 2)),\n        viol_pct=(\'comfort_viol_flag\', lambda s: round(float(s.mean() * 100), 2)),\n        sat_std=(\'eff_sat\', lambda s: round(float(s.std()), 3)),\n        press_std=(\'eff_press\', lambda s: round(float(s.std()), 3)),\n        recirc_std=(\'eff_recirc\', lambda s: round(float(s.std()), 3)),\n        water_std=(\'eff_water\', lambda s: round(float(s.std()), 3)),\n    )\n    pol_stats[\'regret_ratio_pct\'] = ((pol_stats[\'rew_n_mean\'] - pol_stats[\'anc_n_mean\']) / (pol_stats[\'anc_n_mean\'].abs() + 1e-6) * 100).round(2)\n    print(pol_stats.to_string())\n\n    # 3. Persistence of Excitation & VIF (when AHU is actively modulating)\n    ortho_act = df_audit[df_audit[\'policy_tag\'].str.startswith(\'ortho_\') & (df_audit[\'base_press\'] > -0.99)].copy()\n    vifs = {}\n    if len(ortho_act) >= 50:\n        d_mat = np.column_stack([\n            ortho_act[\'eff_sat\'] - ortho_act[\'base_sat\'],\n            ortho_act[\'eff_press\'] - ortho_act[\'base_press\'],\n            ortho_act[\'eff_recirc\'] - ortho_act[\'base_recirc\'],\n            ortho_act[\'eff_water\'] - ortho_act[\'base_water\'],\n            ortho_act[\'oat_celsius\'],\n        ])\n        vif_vals = _compute_vif(d_mat)\n        cols = [\'d_sat\', \'d_press\', \'d_recirc\', \'d_water\', \'oat_C\']\n        vifs = {c: round(float(v), 3) for c, v in zip(cols, vif_vals)}\n        print(\'\\n3. KIEM TRA KICH THICH DOC LAP (Excitation VIF < 5.0 = Khong da cong tuyen):\')\n        print(\'   VIF:\', vifs)\n\n    # 4. BOPTEST KPI Breakdown by Climate Station (Anchor)\n    anc = df_audit[df_audit[\'policy_tag\'] == \'anchor_empty_day\'].copy()\n    anc[\'station\'] = anc[\'scenario\'].str.split(\'/\').str[1].str.rsplit(\'_\', n=1).str[0]\n    clim = anc.groupby(\'station\').agg(\n        rew_per_step=(\'reward_1step\', lambda s: round(float(s.mean()), 5)),\n        comfort_regret=(\'comfort_regret\', lambda s: round(float(s.mean()), 5)),\n        energy_cost=(\'energy_cost\', lambda s: round(float(s.mean()), 5)),\n        carbon_emission=(\'carbon_emission\', lambda s: round(float(s.mean()), 5)),\n        viol_pct=(\'comfort_viol_flag\', lambda s: round(float(s.mean() * 100), 2)),\n    )\n    print(\'\\n4. PHAN RA BOPTEST KPI CUA ANCHOR THEO KHI HAU:\')\n    print(clim.to_string())\n\n    audit_summary = {\n        \'n_rl_samples\': int(len(df_audit)),\n        \'n_thermal_samples\': int(len(df_th)),\n        \'vifs\': vifs,\n    }\n\n    if out_dir is not None:\n        out_p = Path(out_dir)\n        out_p.mkdir(parents=True, exist_ok=True)\n        (out_p / \'data_audit_v2_metrics.json\').write_text(json.dumps(audit_summary, indent=2))\n\n        fig, axes = plt.subplots(2, 2, figsize=(14, 9))\n        # Panel 1: Hour-of-week reward comparison\n        anc_hr = anc.copy()\n        anc_hr[\'hr\'] = (anc_hr[\'step\'] / 12.0).astype(int)\n        hr_s = anc_hr.groupby(\'hr\')[[\'comfort_regret\', \'energy_cost\', \'carbon_emission\']].mean()\n        axes[0, 0].plot(hr_s.index, hr_s[\'comfort_regret\'], label=\'Comfort Regret\')\n        axes[0, 0].plot(hr_s.index, hr_s[\'energy_cost\'], label=\'Energy Cost\')\n        axes[0, 0].plot(hr_s.index, hr_s[\'carbon_emission\'], label=\'Carbon\')\n        axes[0, 0].axvline(72, color=\'red\', linestyle=\'--\', alpha=0.6, label=\'864-step cutoff (72h)\')\n        axes[0, 0].set_title(\'BOPTEST KPI Breakdown across Full Horizon (Hours)\')\n        axes[0, 0].legend(fontsize=8)\n\n        # Panel 2: Action spread across policies (Coverage check)\n        pol_stds = pol_stats[[\'sat_std\', \'press_std\', \'recirc_std\', \'water_std\']]\n        pol_stds.plot(kind=\'bar\', ax=axes[0, 1])\n        axes[0, 1].set_title(\'Action Std by Policy (State-Action Coverage)\')\n        axes[0, 1].tick_params(axis=\'x\', rotation=25)\n\n        # Panel 3: RLS tail > 0.3K by Hour-of-Day\n        if len(df_th) > 0:\n            th_tmp = df_th.copy()\n            th_tmp[\'hr24\'] = ((np.arctan2(th_tmp[\'hod_sin\'], th_tmp[\'hod_cos\']) % (2 * np.pi)) * 12.0 / np.pi).astype(int)\n            th_h = th_tmp.groupby(\'hr24\')[\'rls_err_K\'].agg(mae=lambda s: s.abs().mean(), gt03=lambda s: (s.abs() > 0.3).mean() * 100)\n            axes[1, 0].bar(th_h.index, th_h[\'gt03\'], color=\'tab:orange\')\n            axes[1, 0].set_title(\'% RLS Thermal Error > 0.3K by Hour of Day\')\n            axes[1, 0].set_xlabel(\'Hour (0-23)\')\n\n        # Panel 4: Climate KPI breakdown\n        clim[[\'comfort_regret\', \'energy_cost\', \'carbon_emission\']].plot(kind=\'bar\', ax=axes[1, 1])\n        axes[1, 1].set_title(\'Mean Per-Step BOPTEST KPI by Climate Station\')\n        axes[1, 1].tick_params(axis=\'x\', rotation=15)\n\n        plt.tight_layout()\n        plt.savefig(out_p / \'data_audit_v2_report.png\', dpi=150)\n        plt.show()\n\n    return audit_summary\n\n\ndef run_collect_and_audit_v2(\n    scenario_paths,\n    load_env_fn,\n    create_meta_fn,\n    pid_cls=None,\n    out_dir=None,\n    steps=2016,\n    n_step=6,\n    gamma=0.96,\n    block_steps=6,\n    include_pid_water=True,\n    seed=42,\n):\n    """Collect multi-policy, orthogonal-excitation, KPI-decomposed v2 dataset & run SOTA audit."""\n    out_p = Path(out_dir) if out_dir is not None else Path.cwd()\n    out_p.mkdir(parents=True, exist_ok=True)\n    tag = f\'v2_s{steps}_n{len(scenario_paths)}\'\n\n    raw_eps, anchor_scores, thermal_all = [], {}, []\n    anchor_rn_map = {}\n    max_zones = 0\n    ep_counter = 0\n\n    modes = [\'anchor\', \'ortho_explore\']\n    if include_pid_water and pid_cls is not None:\n        modes.append(\'pid_water\')\n\n    for sc_idx, sc_path in enumerate(tqdm(scenario_paths, desc=f\'Collecting v2 ({steps} steps)\')):\n        bldg = sc_path.parts[-2]\n        sc_key = f\'{bldg}/{sc_path.parts[-1]}\'\n        for m_idx, mode in enumerate(modes):\n            ep_data = _rollout_episode_v2(\n                sc_path, bldg, sc_key, steps, mode, load_env_fn, create_meta_fn,\n                pid_cls=pid_cls, block_steps=block_steps, seed=seed + sc_idx * 17 + m_idx,\n            )\n            ep_data[\'ep_id\'] = ep_counter\n            ep_counter += 1\n            raw_eps.append(ep_data)\n            thermal_all.extend(ep_data[\'th_rows\'])\n            max_zones = max(max_zones, ep_data[\'z_l\'][0].shape[0])\n            if mode == \'anchor\':\n                anchor_scores[sc_key] = float(sum(ep_data[\'r_l\']))\n                rl = ep_data[\'r_l\']\n                anchor_rn_map[sc_key] = [\n                    sum((gamma ** k) * rl[t + k] for k in range(n_step))\n                    for t in range(len(rl) - n_step)\n                ]\n\n    Z_c, M_c, G_c, XB_c, XE_c = [], [], [], [], []\n    Z_ex, G_ex = [], []\n    R_n, R_anc, R_cmf, R_en, R_cb = [], [], [], [], []\n    Z_n, M_n, G_n, XB_n = [], [], [], []\n    B_id, EP_id, ST_idx, ANC, ACT, TAGS = [], [], [], [], [], []\n    pq_rows = []\n\n    for ep in raw_eps:\n        bldg, sc_key, mode = ep[\'bldg\'], ep[\'sc_key\'], ep[\'mode\']\n        T = len(ep[\'r_l\'])\n        nz = ep[\'z_l\'][0].shape[0]\n        mask = np.zeros((max_zones, 1), dtype=np.float32)\n        mask[:nz, 0] = 1.0\n        is_anc = int(mode == \'anchor\')\n\n        zl, gl, xbl, xel = ep[\'z_l\'], ep[\'g_l\'], ep[\'xb_l\'], ep[\'xe_l\']\n        zexl, gexl = ep[\'z_ex_l\'], ep[\'g_ex_l\']\n        rl, regl, enl, cbl, tagsl = ep[\'r_l\'], ep[\'reg_l\'], ep[\'en_l\'], ep[\'cb_l\'], ep[\'tags_l\']\n\n        for t in range(T - n_step):\n            rn = sum((gamma ** k) * rl[t + k] for k in range(n_step))\n            rcmf = sum((gamma ** k) * regl[t + k] for k in range(n_step))\n            ren = sum((gamma ** k) * enl[t + k] for k in range(n_step))\n            rcb = sum((gamma ** k) * cbl[t + k] for k in range(n_step))\n            ra = anchor_rn_map[sc_key][t]\n            is_act = int(xbl[t][1] > -0.99 or xel[t][1] > -0.99 or abs(rl[t]) > 1e-5 or abs(rn) > 1e-5)\n\n            zc = np.zeros((max_zones, ZONE_FEAT_DIM), dtype=np.float32); zc[:nz] = zl[t]\n            zn = np.zeros((max_zones, ZONE_FEAT_DIM), dtype=np.float32); zn[:nz] = zl[t + n_step]\n            ze = np.zeros((max_zones, 3), dtype=np.float32); ze[:nz] = zexl[t]\n\n            Z_c.append(zc); M_c.append(mask); G_c.append(gl[t]); XB_c.append(xbl[t]); XE_c.append(xel[t])\n            Z_ex.append(ze); G_ex.append(gexl[t])\n            R_n.append(rn); R_anc.append(ra); R_cmf.append(rcmf); R_en.append(ren); R_cb.append(rcb)\n            Z_n.append(zn); M_n.append(mask); G_n.append(gl[t + n_step]); XB_n.append(xbl[t + n_step])\n            B_id.append(bldg); EP_id.append(ep[\'ep_id\']); ST_idx.append(t)\n            ANC.append(is_anc); ACT.append(is_act); TAGS.append(tagsl[t])\n\n            oat_c = (float(gl[t][2]) * 15.0 + 290.0) - 273.15\n            pq_rows.append([\n                bldg, sc_key, ep[\'ep_id\'], tagsl[t], is_anc, is_act, t,\n                float(rl[t]), float(rn), float(ra), float(rn - ra),\n                float(regl[t]), float(enl[t]), float(cbl[t]), float(gexl[t][3]),\n                float(xbl[t][0]), float(xel[t][0]), float(xbl[t][1]), float(xel[t][1]),\n                float(xbl[t][2]), float(xel[t][2]), float(xbl[t][3]), float(xel[t][3]),\n                oat_c, float(gl[t][7]), float(gl[t][9]),\n                float(gexl[t][0]), float(gexl[t][1]), float(gexl[t][2]), float(np.mean(zexl[t][:, 2])),\n            ])\n\n    BUF = {\n        \'z_cur\': np.array(Z_c, dtype=np.float32), \'m_cur\': np.array(M_c, dtype=np.float32),\n        \'g_cur\': np.array(G_c, dtype=np.float32), \'xb_cur\': np.array(XB_c, dtype=np.float32),\n        \'xe_cur\': np.array(XE_c, dtype=np.float32), \'r_n\': np.array(R_n, dtype=np.float32)[:, None],\n        \'r_anc\': np.array(R_anc, dtype=np.float32)[:, None],\n        \'z_nxt\': np.array(Z_n, dtype=np.float32), \'m_nxt\': np.array(M_n, dtype=np.float32),\n        \'g_nxt\': np.array(G_n, dtype=np.float32), \'xb_nxt\': np.array(XB_n, dtype=np.float32),\n        \'bldg\': np.array(B_id), \'is_anchor\': np.array(ANC, dtype=np.int32), \'is_active\': np.array(ACT, dtype=np.int32),\n        # v2 SOTA additions\n        \'ep_id\': np.array(EP_id, dtype=np.int32), \'step_idx\': np.array(ST_idx, dtype=np.int32),\n        \'policy_tag\': np.array(TAGS),\n        \'z_extra_cur\': np.array(Z_ex, dtype=np.float32), \'g_extra_cur\': np.array(G_ex, dtype=np.float32),\n        \'r_comfort_n\': np.array(R_cmf, dtype=np.float32)[:, None],\n        \'r_energy_n\': np.array(R_en, dtype=np.float32)[:, None],\n        \'r_carbon_n\': np.array(R_cb, dtype=np.float32)[:, None],\n    }\n\n    npz_path = out_p / f\'warp_replay_{tag}.npz\'\n    pq_path = out_p / f\'warp_transitions_{tag}.parquet\'\n    th_path = out_p / f\'thermal_residuals_{tag}.parquet\'\n    bs_path = out_p / f\'base_scores_{tag}.json\'\n\n    np.savez_compressed(npz_path, **BUF)\n    bs_path.write_text(json.dumps(anchor_scores, indent=2))\n\n    df_audit = pd.DataFrame(pq_rows, columns=[\n        \'building\', \'scenario\', \'ep_id\', \'policy_tag\', \'is_anchor\', \'is_active\', \'step\',\n        \'reward_1step\', \'reward_nstep\', \'reward_anchor\', \'regret_vs_anchor\',\n        \'comfort_regret\', \'energy_cost\', \'carbon_emission\', \'comfort_viol_flag\',\n        \'base_sat\', \'eff_sat\', \'base_press\', \'eff_press\', \'base_recirc\', \'eff_recirc\', \'base_water\', \'eff_water\',\n        \'oat_celsius\', \'occ_norm\', \'heat_recency\',\n        \'dow_sin\', \'dow_cos\', \'d_oat_30m\', \'mean_ahu_spread\',\n    ])\n    df_audit.to_parquet(pq_path, engine=\'pyarrow\', index=False)\n\n    df_th = pd.DataFrame(thermal_all, columns=[\n        \'building\', \'scenario\', \'policy_tag\', \'step\', \'zone_idx\',\n        \'hod_sin\', \'hod_cos\', \'occ_norm\', \'oat_diff\', \'slow_diff\', \'group_diff\', \'mode\',\n        \'power\', \'model_error\', \'residual\', \'theta_gain\', \'zone_area\',\n        \'zone_ua_norm\', \'dT_30m\', \'temp_spread_ahu\', \'dow_sin\', \'dow_cos\', \'d_oat_30m\',\n        \'old_temp\', \'rls_pred\', \'true_temp\', \'rls_err_K\',\n    ])\n    df_th.to_parquet(th_path, engine=\'pyarrow\', index=False)\n\n    audit_summary = audit_dataset_v2(BUF, df_audit, df_th, out_dir=out_p)\n    print(f\'\\nDa luu bo Data v2: {npz_path.name} ({len(df_audit)} RL, {len(df_th)} Thermal)\')\n    return BUF, df_audit, df_th, anchor_scores, audit_summary\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=\'Collect and audit v2 Smart Buildings dataset\')\n    parser.add_argument(\'--steps\', type=int, default=2016)\n    parser.add_argument(\'--max-scenarios\', type=int, default=0, help=\'0 = all 30 scenarios\')\n    parser.add_argument(\'--out-dir\', type=str, default=\'.\')\n    parser.add_argument(\'--no-pid\', action=\'store_true\')\n    args = parser.parse_args()\n\n    from smart_buildings.evaluate import load_hybrid_environment, create_building_meta_data\n    from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy\n\n    train_dir = Path(\'smart_buildings/training_scenarios\')\n    scs = sorted([p.parent for p in train_dir.glob(\'*/*/config.gin\')])\n    if args.max_scenarios > 0:\n        scs = scs[:args.max_scenarios]\n\n    run_collect_and_audit_v2(\n        scs, load_hybrid_environment, create_meta_fn=create_building_meta_data,\n        pid_cls=None if args.no_pid else PIDWaterResetPolicy,\n        out_dir=args.out_dir, steps=args.steps, include_pid_water=not args.no_pid,\n    )\n')
    print('Da ghi physics_foundation.py, warp_resip.py, collect_data_v2.py vao /kaggle/working')
else:
    print('Chay local: su dung truc tiep module trong repo')


In [ ]:
# Cell 1: Cau hinh & Kiem tra Moi truong (Local CPU hoac Kaggle CPU)
import os, sys, json
from pathlib import Path
import numpy as np

# --- CAU HINH DATA V2 ---
STEPS_COLLECT = 2016       # 7 ngay day du (Sun -> Sat). Co the ha xuong 864 neu muon test nhanh, hoac 8640 cho full 30 ngay
MAX_SCENARIOS = None       # Dat so nguyen (vd: 6 = 1 kich ban/toa) de smoke test nhanh; None = full 30 kich ban
INCLUDE_PID_WATER = True   # Thu them luot PIDWaterResetPolicy de mo rong State-Action Coverage
BLOCK_STEPS = 6            # 30 phut giu kich thich don truc
N_STEP = 6
GAMMA = 0.96
SEED = 42

candidates = [
    Path('/kaggle/input/competitions/neurips-2026-smart-buildings'),
    Path('/kaggle/input/neurips-2026-smart-buildings'),
]
DATA_ROOT = next((p for p in candidates if (p / 'smart_buildings').exists()), None)
if DATA_ROOT is None:
    for base in [Path.cwd()] + list(Path.cwd().parents):
        if (base / 'smart_buildings').exists():
            DATA_ROOT = base
            break
assert DATA_ROOT is not None, 'Khong tim thay smart_buildings package!'

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else (DATA_ROOT / 'data_v2_out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [OUT_DIR, Path.cwd(), DATA_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

os.chdir(DATA_ROOT)
from smart_buildings.evaluate import load_hybrid_environment, create_building_meta_data
from smart_buildings.baselines.pid_plus import PIDWaterResetPolicy
from collect_data_v2 import run_collect_and_audit_v2

TRAIN_DIR = DATA_ROOT / 'smart_buildings' / 'training_scenarios'
ALL_SCENARIOS = sorted([p.parent for p in TRAIN_DIR.glob('*/*/config.gin')])
if MAX_SCENARIOS is not None:
    ALL_SCENARIOS = ALL_SCENARIOS[:int(MAX_SCENARIOS)]

print(f'DATA_ROOT : {DATA_ROOT}')
print(f'OUT_DIR   : {OUT_DIR}')
print(f'Kich ban  : {len(ALL_SCENARIOS)} | Steps: {STEPS_COLLECT} | PID_WATER: {INCLUDE_PID_WATER}')


In [ ]:
# Cell 2: Chay Thu thap v2 & In Bao cao Kiem dinh Chat luong Data (SOTA Audit)
BUF_v2, df_audit_v2, df_th_v2, anchor_scores_v2, audit_summary = run_collect_and_audit_v2(
    ALL_SCENARIOS,
    load_hybrid_environment,
    create_building_meta_data,
    pid_cls=PIDWaterResetPolicy if INCLUDE_PID_WATER else None,
    out_dir=OUT_DIR,
    steps=STEPS_COLLECT,
    n_step=N_STEP,
    gamma=GAMMA,
    block_steps=BLOCK_STEPS,
    include_pid_water=INCLUDE_PID_WATER,
    seed=SEED,
)

print('\nCac file nhe co the copy vao experiment_logs/wave_data_v2/ de push GitHub:')
for f in sorted(OUT_DIR.glob('*.json')) + sorted(OUT_DIR.glob('*.png')):
    print(' -', f)
